# Bibliotecas

In [1]:
# Bibliotecas gerais
import os
import time
from datetime import datetime
from getpass import getpass
from IPython.display import display
import html
from pathlib import Path
import math
import warnings
warnings.filterwarnings("ignore")

# Bibliotecas de dados
import pandas as pd
pd.set_option('display.precision', 10)
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Biblioteca para armazenar dados no banco de dados
from sqlalchemy import create_engine

# Configurações banco de dados

In [2]:
# Tenta recuperar a conexão com o banco, se definido antes
db_engine = globals().get("db_engine", None)

# Configuração do MySQL 
if not db_engine:
    # Host e Database
    host = "localhost"
    database = "MF"
    
    # Solicita usuário e senha de forma segura
    user = input("Digite seu usuário MySQL: ")
    password = getpass.getpass("Digite sua senha MySQL (não será exibida): ")
    
    # Cria a engine SQLAlchemy
    db_engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")

    print("Configuração do Banco de Dados criada com Sucesso!!!")

else:
    print("Configuração já existe. Usando configuração já existente!!!")

Digite seu usuário MySQL:  tomida
Digite sua senha MySQL (não será exibida):  ········


Configuração do Banco de Dados criada com Sucesso!!!


# 1) Configurações gerais

In [3]:
%%time
# ==========================================================
# ETAPA 1 - CONFIGURAÇÃO INICIAL DO PROJETO
# Preparação da base + estrutura de pastas + funções auxiliares
# ==========================================================

# =========================
# CONFIGURAÇÕES GERAIS
# =========================
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

# =========================
# ESTRUTURA DE PASTAS
# =========================
PASTA_RESULTADOS = Path("resultados")
PASTA_GRAFICOS = PASTA_RESULTADOS / "graficos"
PASTA_TABELAS = PASTA_RESULTADOS / "tabelas"

PASTA_RESULTADOS.mkdir(exist_ok=True)
PASTA_GRAFICOS.mkdir(exist_ok=True)
PASTA_TABELAS.mkdir(exist_ok=True)

print("Pastas criadas/verificadas com sucesso:")
print(f" - {PASTA_RESULTADOS}")
print(f" - {PASTA_GRAFICOS}")
print(f" - {PASTA_TABELAS}")

# =========================
# FUNÇÕES AUXILIARES DE SALVAMENTO
# =========================
def salvar_tabela_csv(df, nome_arquivo, index=False, sep=";"):
    """
    Descrição:
        Salva um DataFrame em CSV na pasta de tabelas do projeto.

    Parâmetros:
        df (pandas.DataFrame):
            DataFrame a ser salvo.
        nome_arquivo (str):
            Nome do arquivo CSV.
        index (bool):
            Se True, salva o índice.
        sep (str):
            Separador do CSV.

    Retorno:
        str:
            Caminho final do arquivo salvo.
    """
    caminho = PASTA_TABELAS / nome_arquivo
    df.to_csv(caminho, index=index, sep=sep, encoding="utf-8-sig")
    print(f"Tabela CSV salva em: {caminho}")
    return str(caminho)


def salvar_tabela(df, nome_arquivo, index=False):

    """
    Descrição:
        Salva um DataFrame em HTML na pasta de tabelas do projeto.

    Parâmetros:
        df (pandas.DataFrame):
            DataFrame a ser salvo.
        nome_arquivo (str):
            Nome do arquivo HTML.
        index (bool):
            Se True, salva o índice.

    Retorno:
        str:
            Caminho final do arquivo salvo.
    """
    
    caminho = PASTA_TABELAS / nome_arquivo
    html = df.to_html(index=index, border=0, classes="display nowrap compact", justify="center")
    
    with open(caminho, "w", encoding="utf-8") as f:
        f.write(html)
    
    print(f"Tabela salva em: {caminho}")
    return str(caminho)


def salvar_grafico(nome_arquivo, dpi=150, bbox_inches="tight"):
    """
    Descrição:
        Salva o gráfico atualmente aberto do matplotlib na pasta de gráficos.

    Parâmetros:
        nome_arquivo (str):
            Nome do arquivo PNG.
        dpi (int):
            Resolução do gráfico.
        bbox_inches (str):
            Ajuste de borda do arquivo.

    Retorno:
        str:
            Caminho final do arquivo salvo.
    """
    caminho = PASTA_GRAFICOS / nome_arquivo
    plt.savefig(caminho, dpi=dpi, bbox_inches=bbox_inches)
    print(f"Gráfico salvo em: {caminho}")
    return str(caminho)

# =========================
# FUNÇÃO PARA PREPARAR A BASE
# =========================
def preparar_base(df, col_ticker="ticker", col_data="data", col_preco="close", col_volume_fin="volume_fin"):
    """
    Descrição:
        Faz a preparação inicial da base histórica de ações para o projeto do efeito janeiro.

    Parâmetros:
        df (pandas.DataFrame):
            Base original contendo ao menos ticker, data, preço e volume financeiro.
        col_ticker (str):
            Nome da coluna de ticker.
        col_data (str):
            Nome da coluna de data.
        col_preco (str):
            Nome da coluna de preço de fechamento.
        col_volume_fin (str):
            Nome da coluna de volume financeiro.

    Processo:
        1. Copia o DataFrame para evitar alteração da base original.
        2. Padroniza nomes principais.
        3. Converte data para datetime.
        4. Remove linhas sem ticker, data ou preço.
        5. Ordena por ticker e data.
        6. Cria colunas auxiliares de ano, mês e ano_mes.

    Retorno:
        pandas.DataFrame:
            Base preparada para as próximas etapas do projeto.
    """

    print("=" * 70)
    print("INICIANDO PREPARAÇÃO DA BASE")
    print("=" * 70)

    df = df.copy()

    # -----------------------------------------
    # Padroniza colunas principais para nomes fixos
    # -----------------------------------------
    rename_map = {
        col_ticker: "ticker",
        col_data: "data",
        col_preco: "preco",
        col_volume_fin: "volume_fin"
    }
    df = df.rename(columns=rename_map)

    # -----------------------------------------
    # Converte data
    # -----------------------------------------
    df["data"] = pd.to_datetime(df["data"], errors="coerce")

    # -----------------------------------------
    # Remove linhas essenciais faltantes
    # -----------------------------------------
    antes = len(df)
    df = df.dropna(subset=["ticker", "data", "preco"])
    depois = len(df)

    print(f"Linhas antes da limpeza: {antes:,}")
    print(f"Linhas após remover nulos essenciais: {depois:,}")
    print(f"Linhas removidas: {antes - depois:,}")

    # -----------------------------------------
    # Garante ticker como string padronizada
    # -----------------------------------------
    df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()

    # -----------------------------------------
    # Ordena por ticker e data
    # -----------------------------------------
    df = df.sort_values(["ticker", "data"]).reset_index(drop=True)

    # -----------------------------------------
    # Colunas auxiliares de tempo
    # -----------------------------------------
    df["ano"] = df["data"].dt.year
    df["mes"] = df["data"].dt.month
    df["ano_mes"] = df["data"].dt.to_period("M").astype(str)

    # -----------------------------------------
    # Diagnóstico rápido
    # -----------------------------------------
    print("-" * 70)
    print("Resumo da base preparada:")
    print(f"Período: {df['data'].min().date()} até {df['data'].max().date()}")
    print(f"Quantidade de tickers: {df['ticker'].nunique():,}")
    print(f"Quantidade de linhas: {len(df):,}")

    colunas_mostrar = [c for c in ["ticker", "data", "preco", "volume_fin", "ano", "mes", "ano_mes"] if c in df.columns]
    display(df[colunas_mostrar].head())

    print("=" * 70)
    print("BASE PREPARADA COM SUCESSO")
    print("=" * 70)

    return df

Pastas criadas/verificadas com sucesso:
 - resultados
 - resultados\graficos
 - resultados\tabelas
CPU times: total: 15.6 ms
Wall time: 3 ms


## Base de cotações das ações

In [4]:
db_engine = globals().get("db_engine", None)

if not db_engine:
    host = "localhost"
    database = "MF"

    user = input("Digite seu usuário MySQL: ")
    password = getpass.getpass("Digite sua senha MySQL (não será exibida): ")

    db_engine = create_engine(
        f"mysql+mysqlconnector://{user}:{password}@{host}/{database}"
    )


query = f"""
SELECT 
    ticker, 
    data,
    close,
    close_adj,
    volume_fin
FROM cot_hist_consolidado 
"""

print("Lendo dados do banco...")

df_base = pd.read_sql(query, con=db_engine)

Lendo dados do banco...


In [5]:
df_base.shape

(1262516, 5)

In [6]:
df_base

,ticker,data,close,close_adj,volume_fin
0,AALR3,2016-10-28,19.200000,18.830400,"122,334,647.000000"
1,AALR3,2016-10-31,18.060000,17.715700,"45,857,231.000000"
2,AALR3,2016-11-01,17.900000,17.557900,"17,676,981.000000"
3,AALR3,2016-11-03,17.990000,17.646600,"11,132,994.000000"
4,AALR3,2016-11-04,17.750000,17.409900,"6,955,112.000000"
...,...,...,...,...,...
1262511,ZAMP3,2025-11-26,3.500000,3.500000,"166,427.000000"
1262512,ZAMP3,2025-11-27,3.510000,3.510000,"187,665.000000"
1262513,ZAMP3,2025-11-28,3.500000,3.500000,"270,254.000000"
1262514,ZAMP3,2025-12-01,3.500000,3.500000,"95,175.000000"


In [7]:
# Preparar base
df_base_preparada = preparar_base(
     df=df_base,
     col_ticker="ticker",
     col_data="data",
     col_preco="close_adj",  
     col_volume_fin="volume_fin"
 )

# Salva tabela
salvar_tabela(df_base_preparada.head(100), "preview_base_preparada.html", index=False)

INICIANDO PREPARAÇÃO DA BASE
Linhas antes da limpeza: 1,262,516
Linhas após remover nulos essenciais: 1,262,516
Linhas removidas: 0
----------------------------------------------------------------------
Resumo da base preparada:
Período: 2010-01-04 até 2025-12-30
Quantidade de tickers: 698
Quantidade de linhas: 1,262,516


,ticker,data,preco,volume_fin,ano,mes,ano_mes
0,AALR3,2016-10-28,18.830400,"122,334,647.000000",2016,10,2016-10
1,AALR3,2016-10-31,17.715700,"45,857,231.000000",2016,10,2016-10
2,AALR3,2016-11-01,17.557900,"17,676,981.000000",2016,11,2016-11
3,AALR3,2016-11-03,17.646600,"11,132,994.000000",2016,11,2016-11
4,AALR3,2016-11-04,17.409900,"6,955,112.000000",2016,11,2016-11


BASE PREPARADA COM SUCESSO
Tabela salva em: resultados\tabelas\preview_base_preparada.html


'resultados\\tabelas\\preview_base_preparada.html'

# 2) Universo Elegível + Liquidez de Novembro

In [8]:
%%time
# ==========================================================
# ETAPA 2 - UNIVERSO ELEGÍVEL + LIQUIDEZ DE NOVEMBRO
# ==========================================================
print("=" * 70)
print("INICIANDO CONSTRUÇÃO DO UNIVERSO ELEGÍVEL")
print("=" * 70)

df = df_base_preparada.copy()

# ----------------------------------------------------------
# 1) LIQUIDEZ MÉDIA DE NOVEMBRO
# ----------------------------------------------------------
print("Calculando liquidez média de novembro...")

df_nov = df[df["mes"] == 11].copy()

liq_nov = (
    df_nov
    .groupby(["ticker", "ano"], as_index=False)
    .agg(liquidez_novembro=("volume_fin", "mean"))
)

print(f"Total de registros de liquidez: {len(liq_nov):,}")

# ----------------------------------------------------------
# 2) DEFINIR DATAS DE ENTRADA E SAÍDA
# ----------------------------------------------------------
print("Definindo datas de entrada e saída por ano...")

datas_entrada = []
datas_saida = []

anos = sorted(df["ano"].unique())

for ano in anos:

    # -----------------------------
    # ENTRADA -> após 15/dez do ano anterior
    # -----------------------------
    df_entrada = df[
        (df["data"] >= pd.Timestamp(f"{ano-1}-12-15")) &
        (df["data"] <= pd.Timestamp(f"{ano-1}-12-31"))
    ]

    if not df_entrada.empty:
        data_entrada = df_entrada["data"].min()
        datas_entrada.append({"ano": ano, "data_entrada": data_entrada})

    # -----------------------------
    # SAÍDA -> último pregão de janeiro
    # -----------------------------
    df_saida = df[
        (df["data"] >= pd.Timestamp(f"{ano}-01-01")) &
        (df["data"] <= pd.Timestamp(f"{ano}-01-31"))
    ]

    if not df_saida.empty:
        data_saida = df_saida["data"].max()
        datas_saida.append({"ano": ano, "data_saida": data_saida})


df_datas_entrada = pd.DataFrame(datas_entrada)
df_datas_saida = pd.DataFrame(datas_saida)

df_datas = df_datas_entrada.merge(df_datas_saida, on="ano", how="inner")

print(f"Total de anos com datas válidas: {len(df_datas):,}")

# ----------------------------------------------------------
# 3) PEGAR PREÇOS NAS DATAS
# ----------------------------------------------------------
print("Extraindo preços de entrada e saída...")

# Merge entrada
df_entrada_precos = df.merge(
    df_datas.rename(columns={"ano": "ano_janela"}),
    left_on="data",
    right_on="data_entrada",
    how="inner"
)

df_entrada_precos = (
    df_entrada_precos[["ticker", "ano_janela", "preco"]]
    .rename(columns={"ano_janela": "ano", "preco": "preco_entrada"})
    .copy()
)


# Merge saída
df_saida_precos = df.merge(
    df_datas.rename(columns={"ano": "ano_janela"}),
    left_on="data",
    right_on="data_saida",
    how="inner"
)

df_saida_precos = (
    df_saida_precos[["ticker", "ano_janela", "preco"]]
    .rename(columns={"ano_janela": "ano", "preco": "preco_saida"})
    .copy()
)

# ----------------------------------------------------------
# 4) MONTAR UNIVERSO ELEGÍVEL
# ----------------------------------------------------------
print("Montando universo elegível...")

df_universo = (
    liq_nov
    .merge(df_entrada_precos, on=["ticker", "ano"], how="inner")
    .merge(df_saida_precos, on=["ticker", "ano"], how="inner")
    .merge(df_datas, on="ano", how="left")
)

print(f"Total de observações elegíveis: {len(df_universo):,}")
print(f"Total de anos: {df_universo['ano'].nunique():,}")
print(f"Total de tickers únicos: {df_universo['ticker'].nunique():,}")

# ----------------------------------------------------------
# 5) DIAGNÓSTICO POR ANO
# ----------------------------------------------------------
print("Gerando tabela de amostra anual...")

tabela_amostra = (
    df_universo
    .groupby("ano")
    .agg(
        n_empresas=("ticker", "nunique"),
        liquidez_media=("liquidez_novembro", "mean")
    )
    .reset_index()
    .sort_values("ano")
)

display(tabela_amostra.head(10))

# ----------------------------------------------------------
# 6) SALVAR RESULTADOS IMPORTANTES
# ----------------------------------------------------------
print("Salvando outputs...")

# Preview da base universo (para debug)
salvar_tabela(df_universo.head(200), "preview_universo_elegivel.html", index=False)

# Tabela importante para o HTML final
salvar_tabela(tabela_amostra, "tabela_amostra_anual.html", index=False)

print("=" * 70)
print("UNIVERSO ELEGÍVEL CONSTRUÍDO COM SUCESSO")
print("=" * 70)

INICIANDO CONSTRUÇÃO DO UNIVERSO ELEGÍVEL
Calculando liquidez média de novembro...
Total de registros de liquidez: 6,355
Definindo datas de entrada e saída por ano...
Total de anos com datas válidas: 15
Extraindo preços de entrada e saída...
Montando universo elegível...
Total de observações elegíveis: 4,328
Total de anos: 15
Total de tickers únicos: 535
Gerando tabela de amostra anual...


,ano,n_empresas,liquidez_media
0,2011,246,"18,689,324.489603"
1,2012,258,"21,316,504.367817"
2,2013,253,"23,715,471.848912"
3,2014,247,"22,342,507.440689"
4,2015,254,"20,163,222.411436"
5,2016,225,"34,106,275.424739"
6,2017,256,"31,123,697.678786"
7,2018,273,"46,732,041.943836"
8,2019,288,"54,599,821.960055"
9,2020,313,"88,097,998.199722"


Salvando outputs...
Tabela salva em: resultados\tabelas\preview_universo_elegivel.html
Tabela salva em: resultados\tabelas\tabela_amostra_anual.html
UNIVERSO ELEGÍVEL CONSTRUÍDO COM SUCESSO
CPU times: total: 703 ms
Wall time: 713 ms


# 3) Classificação + Retornos + Carteiras

In [9]:
%%time
# ==========================================================
# ETAPA 3 - CLASSIFICAÇÃO + RETORNOS + CARTEIRAS
# ==========================================================
print("=" * 70)
print("INICIANDO CLASSIFICAÇÃO E CÁLCULO DE RETORNOS")
print("=" * 70)

df = df_universo.copy()

# ----------------------------------------------------------
# 1) CLASSIFICAÇÃO POR LIQUIDEZ (POR ANO)
# ----------------------------------------------------------
print("Classificando small vs large caps...")

def classificar_grupo(df_ano):
    df_ano = df_ano.sort_values("liquidez_novembro")

    n = len(df_ano)
    n_cut = int(n * 0.3)

    df_ano["grupo"] = "middle"

    if n_cut > 0:
        df_ano.iloc[:n_cut, df_ano.columns.get_loc("grupo")] = "small"
        df_ano.iloc[-n_cut:, df_ano.columns.get_loc("grupo")] = "large"

    return df_ano

df = (
    df
    .groupby("ano", group_keys=False)
    .apply(classificar_grupo)
)

print("Classificação concluída.")

# ----------------------------------------------------------
# 2) CALCULAR RETORNO
# ----------------------------------------------------------
print("Calculando retornos...")

df["retorno"] = (df["preco_saida"] / df["preco_entrada"]) - 1

# Remover valores absurdos (proteção)
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=["retorno"])

print("Retornos calculados.")

# ----------------------------------------------------------
# 3) FILTRAR APENAS SMALL E LARGE
# ----------------------------------------------------------
df_filtrado = df[df["grupo"].isin(["small", "large"])].copy()

print(f"Total após filtro small/large: {len(df_filtrado):,}")

# ----------------------------------------------------------
# 4) CARTEIRAS POR ANO
# ----------------------------------------------------------
print("Calculando carteiras por ano...")

df_carteiras = (
    df_filtrado
    .groupby(["ano", "grupo"])
    .agg(
        retorno_medio=("retorno", "mean"),
        n_empresas=("ticker", "nunique")
    )
    .reset_index()
)

# Pivot para formato final
df_carteiras_pivot = df_carteiras.pivot(index="ano", columns="grupo", values="retorno_medio")
df_qtd = df_carteiras.pivot(index="ano", columns="grupo", values="n_empresas")

df_resultado = df_carteiras_pivot.copy()
df_resultado["spread_small_minus_large"] = df_resultado["small"] - df_resultado["large"]
df_resultado["n_small"] = df_qtd["small"]
df_resultado["n_large"] = df_qtd["large"]

df_resultado = df_resultado.reset_index().sort_values("ano")

print("Carteiras calculadas.")

# ----------------------------------------------------------
# 5) DIAGNÓSTICO
# ----------------------------------------------------------
print("-" * 70)
print("Resumo das carteiras:")
display(df_resultado.head(10))

# ----------------------------------------------------------
# 6) SALVAR RESULTADOS IMPORTANTES
# ----------------------------------------------------------
print("Salvando outputs...")

# Tabela principal do projeto
tabela_retorno_carteiras_export = df_resultado.copy()
tabela_retorno_carteiras_export.columns.name = None
tabela_retorno_carteiras_export.index.name = None

salvar_tabela(tabela_retorno_carteiras_export, "tabela_retorno_carteiras.html", index=False)

# Preview detalhado (debug)
salvar_tabela(df_filtrado.head(300), "preview_retornos_acoes.html", index=False)

print("=" * 70)
print("ETAPA 3 FINALIZADA COM SUCESSO")
print("=" * 70)

INICIANDO CLASSIFICAÇÃO E CÁLCULO DE RETORNOS
Classificando small vs large caps...
Classificação concluída.
Calculando retornos...
Retornos calculados.
Total após filtro small/large: 2,580
Calculando carteiras por ano...
Carteiras calculadas.
----------------------------------------------------------------------
Resumo das carteiras:


grupo,ano,large,small,spread_small_minus_large,n_small,n_large
0,2011,-0.031594,-0.025558,0.006036,73,73
1,2012,0.085510,0.090151,0.004641,77,77
2,2013,0.029233,0.012623,-0.016610,75,75
3,2014,-0.048711,-0.000309,0.048402,74,74
4,2015,-0.022156,-0.012170,0.009986,76,76
5,2016,-0.096363,-0.027754,0.068610,67,67
6,2017,0.126276,0.203522,0.077246,76,76
7,2018,0.132813,0.083854,-0.048959,81,81
8,2019,0.164028,0.196234,0.032206,86,86
9,2020,0.069710,0.217661,0.147951,93,93


Salvando outputs...
Tabela salva em: resultados\tabelas\tabela_retorno_carteiras.html
Tabela salva em: resultados\tabelas\preview_retornos_acoes.html
ETAPA 3 FINALIZADA COM SUCESSO
CPU times: total: 93.8 ms
Wall time: 90.7 ms


# 4) Janelas de Controle + Base Consolidada

In [10]:
%%time
# ==========================================================
# ETAPA 4 - JANELAS DE CONTROLE + BASE CONSOLIDADA
# ==========================================================
print("=" * 70)
print("INICIANDO ETAPA 4 - JANELAS DE CONTROLE")
print("=" * 70)

df = df_base_preparada.copy()

# ----------------------------------------------------------
# 1) FUNÇÃO AUXILIAR PARA OBTER DATA DE ENTRADA E SAÍDA
# ----------------------------------------------------------
def obter_datas_janela(df, ano, mes_entrada, mes_saida, usar_ano_anterior_na_entrada=False):
    """
    Descrição:
        Obtém a data de entrada e a data de saída de uma janela de análise.

    Regras:
        - entrada = primeiro pregão após o dia 15 do mês de entrada
        - saída   = último pregão do mês de saída

    Parâmetros:
        df (pandas.DataFrame):
            Base preparada com coluna 'data'
        ano (int):
            Ano de referência da janela
        mes_entrada (int):
            Mês da entrada
        mes_saida (int):
            Mês da saída
        usar_ano_anterior_na_entrada (bool):
            Se True, a data de entrada será buscada no ano anterior.
            Isso será usado apenas para a janela dez->jan.

    Retorno:
        tuple:
            (data_entrada, data_saida)
    """

    ano_entrada = ano - 1 if usar_ano_anterior_na_entrada else ano

    df_entrada = df[
        (df["data"] >= pd.Timestamp(year=ano_entrada, month=mes_entrada, day=15)) &
        (df["data"] <= (pd.Timestamp(year=ano_entrada, month=mes_entrada, day=1) + pd.offsets.MonthEnd(0)))
    ].copy()

    df_saida = df[
        (df["data"] >= pd.Timestamp(year=ano, month=mes_saida, day=1)) &
        (df["data"] <= (pd.Timestamp(year=ano, month=mes_saida, day=1) + pd.offsets.MonthEnd(0)))
    ].copy()

    data_entrada = df_entrada["data"].min() if not df_entrada.empty else pd.NaT
    data_saida = df_saida["data"].max() if not df_saida.empty else pd.NaT

    return data_entrada, data_saida

# ----------------------------------------------------------
# 2) FUNÇÃO PARA CONSTRUIR UMA JANELA
# ----------------------------------------------------------
def construir_janela_retorno(
    df_base,
    liq_nov_base,
    mes_entrada,
    mes_saida,
    nome_janela,
    usar_ano_anterior_na_entrada=False
):
    """
    Descrição:
        Constrói a base detalhada de uma janela específica e a tabela
        anual de carteiras small vs large.

    Parâmetros:
        df_base (pandas.DataFrame):
            Base preparada com ticker, data, preco e volume_fin.
        liq_nov_base (pandas.DataFrame):
            Base de liquidez média de novembro por ticker e ano.
        mes_entrada (int):
            Mês da entrada.
        mes_saida (int):
            Mês da saída.
        nome_janela (str):
            Nome padronizado da janela.
        usar_ano_anterior_na_entrada (bool):
            Usado apenas para a janela dez->jan.

    Retorno:
        tuple:
            (df_detalhado_janela, df_carteiras_janela)
    """

    registros_datas = []

    anos = sorted(df_base["ano"].unique())

    # -----------------------------------------
    # Datas da janela por ano
    # -----------------------------------------
    for ano in anos:
        data_entrada, data_saida = obter_datas_janela(
            df=df_base,
            ano=ano,
            mes_entrada=mes_entrada,
            mes_saida=mes_saida,
            usar_ano_anterior_na_entrada=usar_ano_anterior_na_entrada
        )

        if pd.notna(data_entrada) and pd.notna(data_saida):
            registros_datas.append({
                "ano": ano,
                "janela": nome_janela,
                "data_entrada": data_entrada,
                "data_saida": data_saida
            })

    df_datas_janela = pd.DataFrame(registros_datas)

    if df_datas_janela.empty:
        return pd.DataFrame(), pd.DataFrame()

    # -----------------------------------------
    # Preços de entrada
    # -----------------------------------------
    df_entrada = df_base.merge(
        df_datas_janela.rename(columns={"ano": "ano_janela"}),
        left_on="data",
        right_on="data_entrada",
        how="inner"
    )

    df_entrada = (
        df_entrada[["ticker", "ano_janela", "preco"]]
        .rename(columns={
            "ano_janela": "ano",
            "preco": "preco_entrada"
        })
        .copy()
    )

    # -----------------------------------------
    # Preços de saída
    # -----------------------------------------
    df_saida = df_base.merge(
        df_datas_janela.rename(columns={"ano": "ano_janela"}),
        left_on="data",
        right_on="data_saida",
        how="inner"
    )

    df_saida = (
        df_saida[["ticker", "ano_janela", "preco"]]
        .rename(columns={
            "ano_janela": "ano",
            "preco": "preco_saida"
        })
        .copy()
    )

    # -----------------------------------------
    # Liquidez de novembro usada para classificar o ano
    # Para todas as janelas do ano X, usamos novembro de X-1
    # -----------------------------------------
    liq_ref = liq_nov_base.copy()
    liq_ref["ano"] = liq_ref["ano"] + 1

    # -----------------------------------------
    # Montagem da base detalhada
    # -----------------------------------------
    df_janela = (
        liq_ref
        .merge(df_entrada, on=["ticker", "ano"], how="inner")
        .merge(df_saida, on=["ticker", "ano"], how="inner")
        .merge(df_datas_janela[["ano", "janela", "data_entrada", "data_saida"]], on="ano", how="left")
    )

    if df_janela.empty:
        return pd.DataFrame(), pd.DataFrame()

    # -----------------------------------------
    # Classificação por liquidez
    # -----------------------------------------
    def classificar_grupo(df_ano):
        df_ano = df_ano.sort_values("liquidez_novembro").copy()
        n = len(df_ano)
        n_cut = int(n * 0.3)

        df_ano["grupo"] = "middle"

        if n_cut > 0:
            df_ano.iloc[:n_cut, df_ano.columns.get_loc("grupo")] = "small"
            df_ano.iloc[-n_cut:, df_ano.columns.get_loc("grupo")] = "large"

        return df_ano

    df_janela = (
        df_janela
        .groupby("ano", group_keys=False)
        .apply(classificar_grupo)
    )

    # -----------------------------------------
    # Retorno
    # -----------------------------------------
    df_janela["retorno"] = (df_janela["preco_saida"] / df_janela["preco_entrada"]) - 1
    df_janela = df_janela.replace([np.inf, -np.inf], np.nan)
    df_janela = df_janela.dropna(subset=["retorno"])

    df_janela = df_janela[df_janela["grupo"].isin(["small", "large"])].copy()

    # -----------------------------------------
    # Carteiras da janela
    # -----------------------------------------
    df_carteiras = (
        df_janela
        .groupby(["ano", "janela", "grupo"])
        .agg(
            retorno_medio=("retorno", "mean"),
            n_empresas=("ticker", "nunique")
        )
        .reset_index()
    )

    if df_carteiras.empty:
        return df_janela, pd.DataFrame()

    pivot_ret = df_carteiras.pivot(index=["ano", "janela"], columns="grupo", values="retorno_medio")
    pivot_n = df_carteiras.pivot(index=["ano", "janela"], columns="grupo", values="n_empresas")

    df_resumo = pivot_ret.copy()
    df_resumo["spread_small_minus_large"] = df_resumo["small"] - df_resumo["large"]
    df_resumo["n_small"] = pivot_n["small"]
    df_resumo["n_large"] = pivot_n["large"]

    df_resumo = df_resumo.reset_index().sort_values(["janela", "ano"])

    return df_janela, df_resumo

# ----------------------------------------------------------
# 3) LIQUIDEZ BASE DE NOVEMBRO
# ----------------------------------------------------------
print("Calculando base de liquidez de novembro...")

df_nov = df[df["mes"] == 11].copy()

liq_nov_base = (
    df_nov
    .groupby(["ticker", "ano"], as_index=False)
    .agg(liquidez_novembro=("volume_fin", "mean"))
)

print(f"Registros de liquidez base: {len(liq_nov_base):,}")

# ----------------------------------------------------------
# 4) DEFINIÇÃO DAS JANELAS
# ----------------------------------------------------------
print("Definindo janelas...")

janelas = [
    {"mes_entrada": 12, "mes_saida": 1, "nome": "dez_jan", "usar_ano_anterior_na_entrada": True},
    {"mes_entrada": 3,  "mes_saida": 4, "nome": "mar_abr", "usar_ano_anterior_na_entrada": False},
    {"mes_entrada": 4,  "mes_saida": 5, "nome": "abr_mai", "usar_ano_anterior_na_entrada": False},
    {"mes_entrada": 7,  "mes_saida": 8, "nome": "jul_ago", "usar_ano_anterior_na_entrada": False},
    {"mes_entrada": 8,  "mes_saida": 9, "nome": "ago_set", "usar_ano_anterior_na_entrada": False},
]

# ----------------------------------------------------------
# 5) CONSTRUIR TODAS AS JANELAS
# ----------------------------------------------------------
print("Construindo bases das janelas...")

lista_detalhada = []
lista_resumo = []

for janela in janelas:
    print(f"Processando janela: {janela['nome']}")

    df_det, df_res = construir_janela_retorno(
        df_base=df,
        liq_nov_base=liq_nov_base,
        mes_entrada=janela["mes_entrada"],
        mes_saida=janela["mes_saida"],
        nome_janela=janela["nome"],
        usar_ano_anterior_na_entrada=janela["usar_ano_anterior_na_entrada"]
    )

    if not df_det.empty:
        lista_detalhada.append(df_det)

    if not df_res.empty:
        lista_resumo.append(df_res)

df_janelas_detalhado = pd.concat(lista_detalhada, ignore_index=True) if lista_detalhada else pd.DataFrame()
df_janelas_resumo = pd.concat(lista_resumo, ignore_index=True) if lista_resumo else pd.DataFrame()

print(f"Base detalhada consolidada: {len(df_janelas_detalhado):,} linhas")
print(f"Base resumo consolidada: {len(df_janelas_resumo):,} linhas")

# ----------------------------------------------------------
# 6) TABELA COMPARAÇÃO DE JANELAS
# ----------------------------------------------------------
print("Gerando tabela comparativa das janelas...")

tabela_comparacao_janelas = (
    df_janelas_resumo
    .groupby("janela", as_index=False)
    .agg(
        retorno_small_medio=("small", "mean"),
        retorno_large_medio=("large", "mean"),
        spread_medio=("spread_small_minus_large", "mean"),
        mediana_spread=("spread_small_minus_large", "median"),
        desvio_spread=("spread_small_minus_large", "std"),
        anos_com_dados=("ano", "nunique")
    )
    .sort_values("janela")
)

display(tabela_comparacao_janelas)

# ----------------------------------------------------------
# 7) SALVAR OUTPUTS IMPORTANTES
# ----------------------------------------------------------
print("Salvando tabelas da etapa 4...")

tabela_base_consolidada_janelas_export = df_janelas_resumo.copy()
tabela_base_consolidada_janelas_export.columns.name = None
tabela_base_consolidada_janelas_export.index.name = None

salvar_tabela(tabela_base_consolidada_janelas_export, "tabela_base_consolidada_janelas.html", index=False)
salvar_tabela(tabela_comparacao_janelas, "tabela_comparacao_janelas.html", index=False)
salvar_tabela(df_janelas_detalhado.head(300), "preview_janelas_detalhado.html", index=False)

print("=" * 70)
print("ETAPA 4 FINALIZADA COM SUCESSO")
print("=" * 70)

INICIANDO ETAPA 4 - JANELAS DE CONTROLE
Calculando base de liquidez de novembro...
Registros de liquidez base: 6,355
Definindo janelas...
Construindo bases das janelas...
Processando janela: dez_jan
Processando janela: mar_abr
Processando janela: abr_mai
Processando janela: jul_ago
Processando janela: ago_set
Base detalhada consolidada: 12,994 linhas
Base resumo consolidada: 75 linhas
Gerando tabela comparativa das janelas...


,janela,retorno_small_medio,retorno_large_medio,spread_medio,mediana_spread,desvio_spread,anos_com_dados
0,abr_mai,0.028512,0.007139,0.021373,0.015050,0.055759,15
1,ago_set,0.005828,-0.005043,0.010870,0.019409,0.043746,15
2,dez_jan,0.064514,0.029516,0.034998,0.012832,0.069419,15
3,jul_ago,0.030711,0.015564,0.015147,0.022729,0.053324,15
4,mar_abr,0.041189,0.043327,-0.002138,-0.010619,0.065953,15


Salvando tabelas da etapa 4...
Tabela salva em: resultados\tabelas\tabela_base_consolidada_janelas.html
Tabela salva em: resultados\tabelas\tabela_comparacao_janelas.html
Tabela salva em: resultados\tabelas\preview_janelas_detalhado.html
ETAPA 4 FINALIZADA COM SUCESSO
CPU times: total: 2.72 s
Wall time: 2.74 s


In [11]:
display(df_janelas_resumo.head(20))
display(df_janelas_resumo.groupby("janela")["spread_small_minus_large"].describe())
display(tabela_comparacao_janelas)

grupo,ano,janela,large,small,spread_small_minus_large,n_small,n_large
0,2011,dez_jan,-0.032215,-0.028858,0.003357,75,75
1,2012,dez_jan,0.073720,0.086551,0.012832,81,81
2,2013,dez_jan,0.027527,0.016087,-0.011440,76,76
3,2014,dez_jan,-0.056566,0.001381,0.057947,75,75
4,2015,dez_jan,-0.044850,-0.019002,0.025849,77,77
5,2016,dez_jan,-0.088068,-0.039431,0.048637,69,69
6,2017,dez_jan,0.131164,0.221131,0.089967,77,77
7,2018,dez_jan,0.128258,0.086252,-0.042006,82,82
8,2019,dez_jan,0.151821,0.109163,-0.042658,88,88
9,2020,dez_jan,0.050129,0.243731,0.193602,94,94


,count,mean,std,min,25%,50%,75%,max
janela,,,,,,,,
abr_mai,15.000000,0.021373,0.055759,-0.103075,-0.004000,0.015050,0.056937,0.128642
ago_set,15.000000,0.010870,0.043746,-0.060223,-0.020608,0.019409,0.038723,0.096486
dez_jan,15.000000,0.034998,0.069419,-0.042658,-0.009742,0.012832,0.073957,0.193602
jul_ago,15.000000,0.015147,0.053324,-0.075442,-0.025145,0.022729,0.043957,0.123521
mar_abr,15.000000,-0.002138,0.065953,-0.107831,-0.043547,-0.010619,0.037195,0.113845


,janela,retorno_small_medio,retorno_large_medio,spread_medio,mediana_spread,desvio_spread,anos_com_dados
0,abr_mai,0.028512,0.007139,0.021373,0.015050,0.055759,15
1,ago_set,0.005828,-0.005043,0.010870,0.019409,0.043746,15
2,dez_jan,0.064514,0.029516,0.034998,0.012832,0.069419,15
3,jul_ago,0.030711,0.015564,0.015147,0.022729,0.053324,15
4,mar_abr,0.041189,0.043327,-0.002138,-0.010619,0.065953,15


# 5) Estatísticas e Testes

In [12]:
%%time
# ==========================================================
# ETAPA 5 - ESTATÍSTICAS E TESTES
# ==========================================================
print("=" * 70)
print("INICIANDO ETAPA 5 - TESTES ESTATÍSTICOS")
print("=" * 70)

df = df_janelas_resumo.copy()

# ----------------------------------------------------------
# 0) VALIDAÇÃO
# ----------------------------------------------------------
colunas_necessarias = ["ano", "janela", "spread_small_minus_large", "small", "large", "n_small", "n_large"]
faltantes = [c for c in colunas_necessarias if c not in df.columns]
if faltantes:
    raise ValueError(f"Colunas ausentes em df_janelas_resumo: {faltantes}")

ordem_janelas = ["dez_jan", "mar_abr", "abr_mai", "jul_ago", "ago_set"]
janelas_controle = [j for j in ordem_janelas if j != "dez_jan"]

df = df.loc[df["janela"].isin(ordem_janelas)].copy()
df["janela"] = pd.Categorical(df["janela"], categories=ordem_janelas, ordered=True)
df = df.sort_values(["janela", "ano"]).reset_index(drop=True)

# ----------------------------------------------------------
# 1) ESTATÍSTICAS DESCRITIVAS POR JANELA
# ----------------------------------------------------------
print("Calculando estatísticas descritivas por janela...")

def calcular_estatisticas_janela(df_janela):
    s = df_janela["spread_small_minus_large"].dropna()
    return pd.Series({
        "media_spread": s.mean(),
        "mediana_spread": s.median(),
        "desvio_spread": s.std(),
        "min_spread": s.min(),
        "max_spread": s.max(),
        "pct_spread_positivo": (s > 0).mean(),
        "n_anos": s.shape[0],
        "retorno_small_medio": df_janela["small"].mean(),
        "retorno_large_medio": df_janela["large"].mean(),
        "n_small_medio": df_janela["n_small"].mean(),
        "n_large_medio": df_janela["n_large"].mean(),
    })

tabela_estatisticas = (
    df.groupby("janela", observed=True)
      .apply(calcular_estatisticas_janela)
      .reset_index()
      .sort_values("janela")
      .reset_index(drop=True)
)

display(tabela_estatisticas)

# ----------------------------------------------------------
# 2) TESTES CONTRA ZERO (mantidos)
# ----------------------------------------------------------
print("Executando testes contra zero por janela...")

def teste_permutacao_sinal(dados, n_perm=10000, seed=42):
    dados = np.asarray(dados, dtype=float)
    dados = dados[np.isfinite(dados)]

    if dados.size == 0:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)
    observado = dados.mean()

    sinais = rng.choice([-1, 1], size=(n_perm, dados.size))
    simulados = (sinais * dados).mean(axis=1)

    # teste unilateral: média observada > 0
    p_valor = (np.sum(simulados >= observado) + 1) / (n_perm + 1)

    return observado, p_valor

resultados_zero = []

for janela in ordem_janelas:
    dados = (
        df.loc[df["janela"] == janela, "spread_small_minus_large"]
          .dropna()
          .astype(float)
          .values
    )

    if len(dados) == 0:
        continue

    # t unilateral contra zero
    t_stat, p_bicaudal = stats.ttest_1samp(dados, popmean=0, nan_policy="omit")
    if np.isnan(t_stat):
        p_t = np.nan
    else:
        p_t = (p_bicaudal / 2) if t_stat > 0 else (1 - p_bicaudal / 2)

    resultados_zero.append({
        "janela": janela,
        "familia_teste": "contra_zero",
        "teste": "t-test unilateral",
        "hipotese_alternativa": "spread > 0",
        "estatistica": t_stat,
        "p_valor": p_t,
        "n": len(dados)
    })

    # teste de proporção unilateral
    n = len(dados)
    sucessos = int((dados > 0).sum())
    p_hat = sucessos / n
    z = (p_hat - 0.5) / np.sqrt(0.25 / n)
    p_prop = 1 - stats.norm.cdf(z)

    resultados_zero.append({
        "janela": janela,
        "familia_teste": "contra_zero",
        "teste": "teste de proporção unilateral",
        "hipotese_alternativa": "P(spread > 0) > 50%",
        "estatistica": z,
        "p_valor": p_prop,
        "n": n
    })

    # permutação unilateral
    obs, p_perm = teste_permutacao_sinal(dados, n_perm=10000, seed=42)
    resultados_zero.append({
        "janela": janela,
        "familia_teste": "contra_zero",
        "teste": "permutação unilateral",
        "hipotese_alternativa": "spread > 0",
        "estatistica": obs,
        "p_valor": p_perm,
        "n": n
    })

tabela_testes_zero = pd.DataFrame(resultados_zero)
display(tabela_testes_zero)

# ----------------------------------------------------------
# 3) TESTE CENTRAL DA HIPÓTESE
#    dez_jan vs janelas de controle (pareado por ano)
# ----------------------------------------------------------
print("Executando teste central da hipótese: dez_jan vs janelas de controle...")

def p_adjust_holm(p_values):
    p_values = np.asarray(p_values, dtype=float)
    ajustados = np.full_like(p_values, np.nan, dtype=float)

    mask = np.isfinite(p_values)
    if mask.sum() == 0:
        return ajustados

    idx_validos = np.where(mask)[0]
    p_validos = p_values[mask]

    ordem = np.argsort(p_validos)
    p_ord = p_validos[ordem]
    m = len(p_ord)

    ajustados_ord = np.empty(m, dtype=float)
    for i, p in enumerate(p_ord):
        ajustados_ord[i] = (m - i) * p

    for i in range(1, m):
        ajustados_ord[i] = max(ajustados_ord[i], ajustados_ord[i - 1])

    ajustados_ord = np.clip(ajustados_ord, 0, 1)

    ajustados_validos = np.empty(m, dtype=float)
    ajustados_validos[ordem] = ajustados_ord
    ajustados[idx_validos] = ajustados_validos
    return ajustados

def wilcoxon_pareado_maior(diff):
    diff = pd.Series(diff).dropna().astype(float)
    diff = diff[np.isfinite(diff)]

    if len(diff) == 0:
        return np.nan, np.nan

    if np.allclose(diff.values, 0):
        return 0.0, 1.0

    try:
        stat, p = stats.wilcoxon(
            diff,
            alternative="greater",
            zero_method="wilcox",
            correction=False,
            mode="auto"
        )
    except ValueError:
        stat, p = np.nan, np.nan

    return stat, p

def permutacao_pareada_maior(diff, n_perm=20000, seed=42):
    diff = np.asarray(pd.Series(diff).dropna().astype(float))
    diff = diff[np.isfinite(diff)]

    if diff.size == 0:
        return np.nan, np.nan

    if np.allclose(diff, 0):
        return 0.0, 1.0

    rng = np.random.default_rng(seed)
    observado = diff.mean()

    sinais = rng.choice([-1, 1], size=(n_perm, diff.size))
    simulados = (sinais * diff).mean(axis=1)

    p = (np.sum(simulados >= observado) + 1) / (n_perm + 1)
    return observado, p

base_dez = (
    df.loc[df["janela"] == "dez_jan", ["ano", "spread_small_minus_large", "small", "large"]]
      .rename(columns={
          "spread_small_minus_large": "spread_dez_jan",
          "small": "ret_small_dez_jan",
          "large": "ret_large_dez_jan"
      })
      .copy()
)

resultados_pareados = []

for janela_ctrl in janelas_controle:
    base_ctrl = (
        df.loc[df["janela"] == janela_ctrl, ["ano", "spread_small_minus_large", "small", "large"]]
          .rename(columns={
              "spread_small_minus_large": f"spread_{janela_ctrl}",
              "small": f"ret_small_{janela_ctrl}",
              "large": f"ret_large_{janela_ctrl}"
          })
          .copy()
    )

    comp = (
        base_dez.merge(base_ctrl, on="ano", how="inner")
                .sort_values("ano")
                .reset_index(drop=True)
    )

    if comp.empty:
        continue

    comp["diff_spread"] = comp["spread_dez_jan"] - comp[f"spread_{janela_ctrl}"]

    stat_w, p_w = wilcoxon_pareado_maior(comp["diff_spread"])
    stat_p, p_p = permutacao_pareada_maior(comp["diff_spread"], n_perm=20000, seed=42)

    resultados_pareados.append({
        "janela_referencia": "dez_jan",
        "janela_controle": janela_ctrl,
        "comparacao": f"dez_jan > {janela_ctrl}",
        "n_pares": len(comp),
        "media_spread_dez_jan": comp["spread_dez_jan"].mean(),
        "media_spread_controle": comp[f"spread_{janela_ctrl}"].mean(),
        "media_diff_spread": comp["diff_spread"].mean(),
        "mediana_diff_spread": comp["diff_spread"].median(),
        "pct_diff_positiva": (comp["diff_spread"] > 0).mean(),
        "wilcoxon_stat": stat_w,
        "wilcoxon_p_valor": p_w,
        "permutacao_stat": stat_p,
        "permutacao_p_valor": p_p
    })

tabela_comparacoes_pareadas = pd.DataFrame(resultados_pareados)

if not tabela_comparacoes_pareadas.empty:
    tabela_comparacoes_pareadas["wilcoxon_p_holm"] = p_adjust_holm(
        tabela_comparacoes_pareadas["wilcoxon_p_valor"].values
    )
    tabela_comparacoes_pareadas["permutacao_p_holm"] = p_adjust_holm(
        tabela_comparacoes_pareadas["permutacao_p_valor"].values
    )

display(tabela_comparacoes_pareadas)

# ----------------------------------------------------------
# 4) RESUMO DA HIPÓTESE CENTRAL
# ----------------------------------------------------------
print("Montando resumo interpretável da hipótese central...")

if not tabela_comparacoes_pareadas.empty:
    tabela_resumo_hipotese_central = tabela_comparacoes_pareadas[[
        "comparacao",
        "n_pares",
        "media_spread_dez_jan",
        "media_spread_controle",
        "media_diff_spread",
        "mediana_diff_spread",
        "pct_diff_positiva",
        "wilcoxon_p_valor",
        "wilcoxon_p_holm",
        "permutacao_p_valor",
        "permutacao_p_holm"
    ]].copy()
else:
    tabela_resumo_hipotese_central = pd.DataFrame()

display(tabela_resumo_hipotese_central)

# ----------------------------------------------------------
# 5) TABELA LONGA CONSOLIDADA DE TESTES
# ----------------------------------------------------------
print("Consolidando tabelas de testes...")

lista_testes_pareados_long = []
for _, row in tabela_comparacoes_pareadas.iterrows():
    lista_testes_pareados_long.append({
        "janela": row["janela_referencia"],
        "janela_controle": row["janela_controle"],
        "familia_teste": "hipotese_central_pareada",
        "teste": "Wilcoxon pareado unilateral",
        "hipotese_alternativa": f"spread(dez_jan) > spread({row['janela_controle']})",
        "estatistica": row["wilcoxon_stat"],
        "p_valor": row["wilcoxon_p_valor"],
        "p_valor_holm": row["wilcoxon_p_holm"],
        "n": row["n_pares"]
    })
    lista_testes_pareados_long.append({
        "janela": row["janela_referencia"],
        "janela_controle": row["janela_controle"],
        "familia_teste": "hipotese_central_pareada",
        "teste": "Permutação pareada unilateral",
        "hipotese_alternativa": f"spread(dez_jan) > spread({row['janela_controle']})",
        "estatistica": row["permutacao_stat"],
        "p_valor": row["permutacao_p_valor"],
        "p_valor_holm": row["permutacao_p_holm"],
        "n": row["n_pares"]
    })

tabela_testes_pareados_long = pd.DataFrame(lista_testes_pareados_long)

tabela_testes = pd.concat(
    [
        tabela_testes_zero.assign(janela_controle=np.nan, p_valor_holm=np.nan),
        tabela_testes_pareados_long
    ],
    ignore_index=True
)

display(tabela_testes)

# ----------------------------------------------------------
# 6) SALVAR RESULTADOS
# ----------------------------------------------------------
print("Salvando tabelas...")

salvar_tabela(tabela_estatisticas, "tabela_resumo_estatistico.html", index=False)
salvar_tabela(tabela_testes_zero, "tabela_testes_contra_zero.html", index=False)
salvar_tabela(tabela_comparacoes_pareadas, "tabela_comparacoes_pareadas_janela.html", index=False)
salvar_tabela(tabela_resumo_hipotese_central, "tabela_resumo_hipotese_central.html", index=False)
salvar_tabela(tabela_testes, "tabela_testes_estatisticos.html", index=False)

print("=" * 70)
print("ETAPA 5 FINALIZADA")
print("=" * 70)

INICIANDO ETAPA 5 - TESTES ESTATÍSTICOS
Calculando estatísticas descritivas por janela...


,janela,media_spread,mediana_spread,desvio_spread,min_spread,max_spread,pct_spread_positivo,n_anos,retorno_small_medio,retorno_large_medio,n_small_medio,n_large_medio
0,dez_jan,0.034998,0.012832,0.069419,-0.042658,0.193602,0.666667,15.000000,0.064514,0.029516,88.066667,88.066667
1,mar_abr,-0.002138,-0.010619,0.065953,-0.107831,0.113845,0.400000,15.000000,0.041189,0.043327,86.066667,86.066667
2,abr_mai,0.021373,0.015050,0.055759,-0.103075,0.128642,0.733333,15.000000,0.028512,0.007139,87.066667,87.066667
3,jul_ago,0.015147,0.022729,0.053324,-0.075442,0.123521,0.600000,15.000000,0.030711,0.015564,86.000000,86.000000
4,ago_set,0.010870,0.019409,0.043746,-0.060223,0.096486,0.666667,15.000000,0.005828,-0.005043,85.933333,85.933333


Executando testes contra zero por janela...


,janela,familia_teste,teste,hipotese_alternativa,estatistica,p_valor,n
0,dez_jan,contra_zero,t-test unilateral,spread > 0,1.952586,0.035579,15
1,dez_jan,contra_zero,teste de proporção unilateral,P(spread > 0) > 50%,1.290994,0.098353,15
2,dez_jan,contra_zero,permutação unilateral,spread > 0,0.034998,0.036296,15
3,mar_abr,contra_zero,t-test unilateral,spread > 0,-0.125564,0.549069,15
4,mar_abr,contra_zero,teste de proporção unilateral,P(spread > 0) > 50%,-0.774597,0.780711,15
5,mar_abr,contra_zero,permutação unilateral,spread > 0,-0.002138,0.545545,15
6,abr_mai,contra_zero,t-test unilateral,spread > 0,1.484572,0.079911,15
7,abr_mai,contra_zero,teste de proporção unilateral,P(spread > 0) > 50%,1.807392,0.035351,15
8,abr_mai,contra_zero,permutação unilateral,spread > 0,0.021373,0.082892,15
9,jul_ago,contra_zero,t-test unilateral,spread > 0,1.100139,0.144914,15


Executando teste central da hipótese: dez_jan vs janelas de controle...


...

Montando resumo interpretável da hipótese central...


,comparacao,n_pares,media_spread_dez_jan,media_spread_controle,media_diff_spread,mediana_diff_spread,pct_diff_positiva,wilcoxon_p_valor,wilcoxon_p_holm,permutacao_p_valor,permutacao_p_holm
0,dez_jan > mar_abr,15,0.034998,-0.002138,0.037136,0.026824,0.733333,0.047302,0.189209,0.079196,0.316784
1,dez_jan > abr_mai,15,0.034998,0.021373,0.013625,0.007082,0.533333,0.488983,0.678772,0.346983,0.415479
2,dez_jan > jul_ago,15,0.034998,0.015147,0.019851,0.036927,0.666667,0.339386,0.678772,0.207190,0.415479
3,dez_jan > ago_set,15,0.034998,0.010870,0.024128,0.038538,0.600000,0.194702,0.584106,0.138493,0.415479


Consolidando tabelas de testes...


,janela,familia_teste,teste,hipotese_alternativa,estatistica,p_valor,n,janela_controle,p_valor_holm
0,dez_jan,contra_zero,t-test unilateral,spread > 0,1.952586,0.035579,15,NaN,NaN
1,dez_jan,contra_zero,teste de proporção unilateral,P(spread > 0) > 50%,1.290994,0.098353,15,NaN,NaN
2,dez_jan,contra_zero,permutação unilateral,spread > 0,0.034998,0.036296,15,NaN,NaN
3,mar_abr,contra_zero,t-test unilateral,spread > 0,-0.125564,0.549069,15,NaN,NaN
4,mar_abr,contra_zero,teste de proporção unilateral,P(spread > 0) > 50%,-0.774597,0.780711,15,NaN,NaN
5,mar_abr,contra_zero,permutação unilateral,spread > 0,-0.002138,0.545545,15,NaN,NaN
6,abr_mai,contra_zero,t-test unilateral,spread > 0,1.484572,0.079911,15,NaN,NaN
7,abr_mai,contra_zero,teste de proporção unilateral,P(spread > 0) > 50%,1.807392,0.035351,15,NaN,NaN
8,abr_mai,contra_zero,permutação unilateral,spread > 0,0.021373,0.082892,15,NaN,NaN
9,jul_ago,contra_zero,t-test unilateral,spread > 0,1.100139,0.144914,15,NaN,NaN


Salvando tabelas...
Tabela salva em: resultados\tabelas\tabela_resumo_estatistico.html
Tabela salva em: resultados\tabelas\tabela_testes_contra_zero.html
Tabela salva em: resultados\tabelas\tabela_comparacoes_pareadas_janela.html
Tabela salva em: resultados\tabelas\tabela_resumo_hipotese_central.html
Tabela salva em: resultados\tabelas\tabela_testes_estatisticos.html
ETAPA 5 FINALIZADA
CPU times: total: 141 ms
Wall time: 142 ms


# 6) Visualizações e Tabelas Finais

In [13]:
%%time
# ==========================================================
# ETAPA 6 - VISUALIZAÇÕES E TABELAS FINAIS
# ==========================================================
print("=" * 70)
print("INICIANDO ETAPA 6 - VISUALIZAÇÕES E TABELAS FINAIS")
print("=" * 70)

# ----------------------------------------------------------
# 0) VALIDAÇÃO
# ----------------------------------------------------------
objetos_necessarios = [
    "df_janelas_resumo",
    "tabela_estatisticas",
    "tabela_testes_zero",
    "tabela_comparacoes_pareadas",
    "tabela_resumo_hipotese_central",
    "tabela_testes"
]

faltantes = [obj for obj in objetos_necessarios if obj not in globals()]
if faltantes:
    raise ValueError(f"Objetos ausentes para a ETAPA 6: {faltantes}. Rode a ETAPA 5 antes.")

df = df_janelas_resumo.copy()

ordem_janelas = ["dez_jan", "mar_abr", "abr_mai", "jul_ago", "ago_set"]
janelas_controle = [j for j in ordem_janelas if j != "dez_jan"]

df = df.loc[df["janela"].isin(ordem_janelas)].copy()
df["janela"] = pd.Categorical(df["janela"], categories=ordem_janelas, ordered=True)
df = df.sort_values(["janela", "ano"]).reset_index(drop=True)

# ----------------------------------------------------------
# 1) DIRETÓRIOS E FUNÇÕES AUXILIARES
# ----------------------------------------------------------
DIR_RESULTADOS_LOCAL = globals().get("DIR_RESULTADOS", Path("resultados"))
DIR_GRAFICOS_LOCAL = globals().get("DIR_GRAFICOS", DIR_RESULTADOS_LOCAL / "graficos")
DIR_TABELAS_LOCAL = globals().get("DIR_TABELAS", DIR_RESULTADOS_LOCAL / "tabelas")

Path(DIR_RESULTADOS_LOCAL).mkdir(parents=True, exist_ok=True)
Path(DIR_GRAFICOS_LOCAL).mkdir(parents=True, exist_ok=True)
Path(DIR_TABELAS_LOCAL).mkdir(parents=True, exist_ok=True)

def salvar_figura(caminho, dpi=150):
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(caminho, dpi=dpi, bbox_inches="tight")
    plt.close()

print("Diretórios validados.")
print(f" - Gráficos: {DIR_GRAFICOS_LOCAL}")
print(f" - Tabelas : {DIR_TABELAS_LOCAL}")

# ----------------------------------------------------------
# 2) BASES AUXILIARES
# ----------------------------------------------------------
print("Preparando bases auxiliares...")

df_ret_medio_grupos = (
    df.groupby("janela", observed=True)[["small", "large"]]
      .mean()
      .reindex(ordem_janelas)
      .reset_index()
)

df_spread_medio = (
    tabela_estatisticas[["janela", "media_spread", "mediana_spread", "pct_spread_positivo", "n_anos"]]
    .copy()
    .sort_values("janela")
    .reset_index(drop=True)
)

df_spread_ano_pivot = (
    df.pivot(index="ano", columns="janela", values="spread_small_minus_large")
      .reindex(columns=ordem_janelas)
      .sort_index()
)

base_dez = (
    df.loc[df["janela"] == "dez_jan", ["ano", "spread_small_minus_large"]]
      .rename(columns={"spread_small_minus_large": "spread_dez_jan"})
      .copy()
)

lista_diffs = []
for janela_ctrl in janelas_controle:
    base_ctrl = (
        df.loc[df["janela"] == janela_ctrl, ["ano", "spread_small_minus_large"]]
          .rename(columns={"spread_small_minus_large": "spread_controle"})
          .copy()
    )

    comp = (
        base_dez.merge(base_ctrl, on="ano", how="inner")
                .sort_values("ano")
                .reset_index(drop=True)
    )

    if comp.empty:
        continue

    comp["janela_controle"] = janela_ctrl
    comp["diff_spread"] = comp["spread_dez_jan"] - comp["spread_controle"]
    lista_diffs.append(comp)

if lista_diffs:
    df_diffs_pareados = pd.concat(lista_diffs, ignore_index=True)
else:
    df_diffs_pareados = pd.DataFrame(
        columns=["ano", "spread_dez_jan", "spread_controle", "janela_controle", "diff_spread"]
    )

df_resumo_pareado_plot = tabela_resumo_hipotese_central.copy()
if not df_resumo_pareado_plot.empty:
    df_resumo_pareado_plot["controle_label"] = (
        df_resumo_pareado_plot["comparacao"]
        .astype(str)
        .str.replace("dez_jan > ", "", regex=False)
    )

df_dez_jan_linha = (
    df.loc[df["janela"] == "dez_jan", ["ano", "spread_small_minus_large"]]
      .dropna()
      .sort_values("ano")
      .reset_index(drop=True)
)

dados_hist_jan = (
    df.loc[df["janela"] == "dez_jan", "spread_small_minus_large"]
      .dropna()
      .values
)

print("Bases auxiliares prontas.")
print(f" - Observações em df_janelas_resumo: {len(df)}")
print(f" - Comparações pareadas: {len(df_resumo_pareado_plot)}")

# ----------------------------------------------------------
# 3) GRÁFICO - RETORNO MÉDIO POR JANELA (SMALL VS LARGE)
# ----------------------------------------------------------
print("Gerando gráfico - retorno médio por janela (small vs large)...")

x = np.arange(len(df_ret_medio_grupos))
largura = 0.38

plt.figure(figsize=(10, 6))
plt.bar(x - largura / 2, df_ret_medio_grupos["small"], width=largura, label="small")
plt.bar(x + largura / 2, df_ret_medio_grupos["large"], width=largura, label="large")
plt.xticks(x, df_ret_medio_grupos["janela"].astype(str))
plt.title("Retorno Médio por Janela (Small vs Large)")
plt.xlabel("Janela")
plt.ylabel("Retorno médio")
plt.legend(title="grupo")
plt.grid(True, axis="y", alpha=0.3)
salvar_figura(Path(DIR_GRAFICOS_LOCAL) / "grafico_retorno_medio_janelas.png")

# ----------------------------------------------------------
# 4) GRÁFICO - BOXPLOT DO SPREAD POR JANELA
# ----------------------------------------------------------
print("Gerando gráfico - boxplot do spread por janela...")

dados_boxplot = [
    df.loc[df["janela"] == janela, "spread_small_minus_large"].dropna().values
    for janela in ordem_janelas
]

plt.figure(figsize=(10, 6))
plt.boxplot(dados_boxplot, labels=ordem_janelas, showmeans=True)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Distribuição do Spread por Janela")
plt.xlabel("Janela")
plt.ylabel("Spread")
plt.grid(True, axis="y", alpha=0.3)
salvar_figura(Path(DIR_GRAFICOS_LOCAL) / "boxplot_spread_janelas.png")

plt.figure(figsize=(10, 6))
plt.boxplot(dados_boxplot, labels=ordem_janelas, showmeans=True)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Boxplot do Spread por Janela")
plt.xlabel("Janela")
plt.ylabel("Spread")
plt.grid(True, axis="y", alpha=0.3)
salvar_figura(Path(DIR_GRAFICOS_LOCAL) / "grafico_boxplot_spread_por_janela.png")

# ----------------------------------------------------------
# 5) GRÁFICO - EVOLUÇÃO TEMPORAL DO SPREAD
# ----------------------------------------------------------
print("Gerando gráfico - evolução temporal do spread...")

plt.figure(figsize=(10, 6))
plt.plot(
    df_dez_jan_linha["ano"],
    df_dez_jan_linha["spread_small_minus_large"],
    marker="o",
    linewidth=2
)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Evolução Temporal do Spread")
plt.xlabel("Ano")
plt.ylabel("Spread")
plt.grid(True, alpha=0.3)
salvar_figura(Path(DIR_GRAFICOS_LOCAL) / "linha_spread_tempo.png")

# ----------------------------------------------------------
# 6) GRÁFICO - HISTOGRAMA DO SPREAD EM DEZEMBRO→JANEIRO
# ----------------------------------------------------------
print("Gerando gráfico - histograma do spread em dezembro→janeiro...")

n_bins = max(5, min(10, len(dados_hist_jan)))

plt.figure(figsize=(10, 6))
plt.hist(dados_hist_jan, bins=n_bins)
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("Distribuição do Spread em Dezembro→Janeiro")
plt.xlabel("Spread")
plt.ylabel("Frequência")
plt.grid(True, axis="y", alpha=0.3)
salvar_figura(Path(DIR_GRAFICOS_LOCAL) / "hist_spread_janeiro.png")

# ----------------------------------------------------------
# 7) GRÁFICO - SPREAD MÉDIO POR JANELA
# ----------------------------------------------------------
print("Gerando gráfico - spread médio por janela...")

plt.figure(figsize=(10, 6))
plt.bar(df_spread_medio["janela"].astype(str), df_spread_medio["media_spread"])
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Spread Médio Small-Minus-Large por Janela")
plt.xlabel("Janela")
plt.ylabel("Spread médio")
plt.grid(True, axis="y", alpha=0.3)

for i, row in df_spread_medio.iterrows():
    plt.text(
        i,
        row["media_spread"],
        f"{row['media_spread']:.3f}",
        ha="center",
        va="bottom" if row["media_spread"] >= 0 else "top"
    )

salvar_figura(Path(DIR_GRAFICOS_LOCAL) / "grafico_spread_medio_por_janela.png")

# ----------------------------------------------------------
# 8) GRÁFICO - SPREAD POR ANO E JANELA
# ----------------------------------------------------------
print("Gerando gráfico - spread por ano e janela...")

plt.figure(figsize=(12, 7))

for janela in ordem_janelas:
    if janela in df_spread_ano_pivot.columns:
        plt.plot(
            df_spread_ano_pivot.index,
            df_spread_ano_pivot[janela],
            marker="o",
            linewidth=1.8,
            label=janela
        )

plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Spread por Ano e Janela")
plt.xlabel("Ano")
plt.ylabel("Spread")
plt.legend()
plt.grid(True, alpha=0.3)
salvar_figura(Path(DIR_GRAFICOS_LOCAL) / "grafico_spread_por_ano_e_janela.png")

# ----------------------------------------------------------
# 9) GRÁFICO - COMPARAÇÃO PAREADA
# ----------------------------------------------------------
print("Gerando gráfico - comparação pareada...")

if not df_resumo_pareado_plot.empty:
    plt.figure(figsize=(10, 6))
    plt.bar(df_resumo_pareado_plot["controle_label"], df_resumo_pareado_plot["media_diff_spread"])
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.title("Comparação Pareada: Dezembro→Janeiro vs Controles")
    plt.xlabel("Janela de controle")
    plt.ylabel("Diferença média do spread")
    plt.grid(True, axis="y", alpha=0.3)

    for i, row in df_resumo_pareado_plot.iterrows():
        texto = (
            f"Δ={row['media_diff_spread']:.3f}\n"
            f"p_w={row['wilcoxon_p_valor']:.3f}\n"
            f"p_holm={row['wilcoxon_p_holm']:.3f}"
        )
        plt.text(
            i,
            row["media_diff_spread"],
            texto,
            ha="center",
            va="bottom" if row["media_diff_spread"] >= 0 else "top",
            fontsize=9
        )

    salvar_figura(Path(DIR_GRAFICOS_LOCAL) / "grafico_comparacao_pareada_dez_jan_vs_controles.png")

# ----------------------------------------------------------
# 10) GRÁFICO - BOXPLOT DAS DIFERENÇAS PAREADAS
# ----------------------------------------------------------
print("Gerando gráfico - boxplot das diferenças pareadas...")

if not df_diffs_pareados.empty:
    dados_boxplot_diff = [
        df_diffs_pareados.loc[
            df_diffs_pareados["janela_controle"] == janela_ctrl,
            "diff_spread"
        ].dropna().values
        for janela_ctrl in janelas_controle
    ]

    plt.figure(figsize=(10, 6))
    plt.boxplot(dados_boxplot_diff, labels=janelas_controle, showmeans=True)
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.title("Distribuição das Diferenças Pareadas")
    plt.xlabel("Janela de controle")
    plt.ylabel("Diferença de spread")
    plt.grid(True, axis="y", alpha=0.3)
    salvar_figura(Path(DIR_GRAFICOS_LOCAL) / "grafico_boxplot_diferencas_pareadas.png")

# ----------------------------------------------------------
# 11) TABELAS FINAIS
# ----------------------------------------------------------
print("Salvando tabelas finais da etapa...")

df_spread_medio_export = df_spread_medio.copy()
df_spread_medio_export.columns.name = None
df_spread_medio_export.index.name = None

df_spread_ano_pivot_export = df_spread_ano_pivot.copy()
df_spread_ano_pivot_export.columns.name = None
df_spread_ano_pivot_export.index.name = "ano"

df_diffs_pareados_export = df_diffs_pareados.copy()
df_diffs_pareados_export.columns.name = None
df_diffs_pareados_export.index.name = None

tabela_resumo_hipotese_central_export = tabela_resumo_hipotese_central.copy()
tabela_resumo_hipotese_central_export.columns.name = None
tabela_resumo_hipotese_central_export.index.name = None

salvar_tabela(df_spread_medio_export, "tabela_spread_medio_por_janela.html", index=False)
salvar_tabela(df_spread_ano_pivot_export.reset_index(), "tabela_spread_por_ano_e_janela.html", index=False)
salvar_tabela(df_diffs_pareados_export, "tabela_diferencas_pareadas_por_ano.html", index=False)
salvar_tabela(tabela_resumo_hipotese_central_export, "tabela_resumo_hipotese_central_visual.html", index=False)

# ----------------------------------------------------------
# 12) REGISTRO FINAL
# ----------------------------------------------------------
globals()["df_ret_medio_grupos"] = df_ret_medio_grupos
globals()["df_spread_medio"] = df_spread_medio
globals()["df_spread_ano_pivot"] = df_spread_ano_pivot
globals()["df_diffs_pareados"] = df_diffs_pareados

print("\nArquivos gerados:")
arquivos_gerados = [
    "grafico_retorno_medio_janelas.png",
    "boxplot_spread_janelas.png",
    "linha_spread_tempo.png",
    "hist_spread_janeiro.png",
    "grafico_spread_medio_por_janela.png",
    "grafico_boxplot_spread_por_janela.png",
    "grafico_spread_por_ano_e_janela.png",
    "grafico_comparacao_pareada_dez_jan_vs_controles.png",
    "grafico_boxplot_diferencas_pareadas.png",
    "tabela_spread_medio_por_janela.html",
    "tabela_spread_por_ano_e_janela.html",
    "tabela_diferencas_pareadas_por_ano.html",
    "tabela_resumo_hipotese_central_visual.html",
]

for nome in arquivos_gerados:
    print(f" - {nome}")

print("=" * 70)
print("ETAPA 6 FINALIZADA")
print("=" * 70)

INICIANDO ETAPA 6 - VISUALIZAÇÕES E TABELAS FINAIS
Diretórios validados.
 - Gráficos: resultados\graficos
 - Tabelas : resultados\tabelas
Preparando bases auxiliares...
Bases auxiliares prontas.
 - Observações em df_janelas_resumo: 75
 - Comparações pareadas: 4
Gerando gráfico - retorno médio por janela (small vs large)...
Gerando gráfico - boxplot do spread por janela...
Gerando gráfico - evolução temporal do spread...
Gerando gráfico - histograma do spread em dezembro→janeiro...
Gerando gráfico - spread médio por janela...
Gerando gráfico - spread por ano e janela...
Gerando gráfico - comparação pareada...
Gerando gráfico - boxplot das diferenças pareadas...
Salvando tabelas finais da etapa...
Tabela salva em: resultados\tabelas\tabela_spread_medio_por_janela.html
Tabela salva em: resultados\tabelas\tabela_spread_por_ano_e_janela.html
Tabela salva em: resultados\tabelas\tabela_diferencas_pareadas_por_ano.html
Tabela salva em: resultados\tabelas\tabela_resumo_hipotese_central_visual.h

# 7) HTML Final

In [14]:
%%time
# ==========================================================
# ETAPA 7 - HTML FINAL
# ==========================================================
print("=" * 70)
print("ETAPA 7 - HTML FINAL")
print("=" * 70)

# ----------------------------------------------------------
# [1/6] VALIDAÇÕES
# ----------------------------------------------------------
print("\n[1/6] Validando ambiente...")

BASE_DIR = Path.cwd()

RESULTADOS_DIR = BASE_DIR / "resultados"
GRAFICOS_DIR = RESULTADOS_DIR / "graficos"
TABELAS_DIR = RESULTADOS_DIR / "tabelas"

if not GRAFICOS_DIR.exists():
    GRAFICOS_DIR = RESULTADOS_DIR

if not TABELAS_DIR.exists():
    TABELAS_DIR = RESULTADOS_DIR

print(f"BASE_DIR......: {BASE_DIR}")
print(f"GRAFICOS_DIR..: {GRAFICOS_DIR}")
print(f"TABELAS_DIR...: {TABELAS_DIR}")
print("OK")

# ----------------------------------------------------------
# [2/6] FUNÇÕES AUXILIARES
# ----------------------------------------------------------
print("\n[2/6] Definindo funções auxiliares...")

def arquivo_existe(pasta, nome, extensao):
    return (Path(pasta) / f"{nome}.{extensao}").exists()

def ler_html_tabela(nome_arquivo):
    caminho = TABELAS_DIR / f"{nome_arquivo}.html"
    if caminho.exists():
        with open(caminho, "r", encoding="utf-8") as f:
            return f.read()
    return ""

def primeira_tabela_existente(candidatos):
    for nome in candidatos:
        conteudo = ler_html_tabela(nome)
        if conteudo.strip():
            return nome, conteudo
    return None, ""

def imagem_html(nome_arquivo, titulo=None):
    caminho = GRAFICOS_DIR / f"{nome_arquivo}.png"
    if not caminho.exists():
        return ""

    caminho_rel = caminho.relative_to(BASE_DIR).as_posix()
    titulo_html = f"<h3>{html.escape(titulo)}</h3>" if titulo else ""

    return f"""
    <div class="chart-card">
        {titulo_html}
        <img src="{caminho_rel}" alt="{html.escape(nome_arquivo)}" class="zoomable-img">
    </div>
    """

def bloco_imagens(imagens):
    html_imgs = "".join([
        imagem_html(nome, titulo)
        for nome, titulo in imagens
        if arquivo_existe(GRAFICOS_DIR, nome, "png")
    ])

    if not html_imgs:
        return ""

    return f'<div class="grid-2">{html_imgs}</div>'

def bloco_imagens_com_titulo(titulo, imagens):
    bloco = bloco_imagens(imagens)
    if not bloco.strip():
        return ""
    return f"""
    <div class="subsection-block">
        <h3>{html.escape(titulo)}</h3>
        {bloco}
    </div>
    """

def bloco_tabelas_html(tabelas):
    html_tabs = ""

    for nome, titulo in tabelas:
        tabela_html = ler_html_tabela(nome)

        if tabela_html.strip():
            html_tabs += f"""
            <div class="table-block">
                <h3>{html.escape(titulo)}</h3>
                <div class="table-wrapper">
                    {tabela_html}
                </div>
            </div>
            """

    return html_tabs

def bloco_tabela_preferencial(candidatos, titulo):
    nome_escolhido, tabela_html = primeira_tabela_existente(candidatos)
    if not tabela_html.strip():
        return ""
    return f"""
    <div class="table-block">
        <h3>{html.escape(titulo)}</h3>
        <div class="table-wrapper">
            {tabela_html}
        </div>
    </div>
    """

def secao_html(titulo, descricao="", conteudo=""):
    if not conteudo.strip():
        return ""
    descricao_html = f"<p class='section-desc'>{descricao}</p>" if descricao else ""
    return f"""
    <section class="section">
        <h2>{html.escape(titulo)}</h2>
        {descricao_html}
        {conteudo}
    </section>
    """

def fmt_pct(x, casas=1):
    if x is None or (isinstance(x, float) and (np.isnan(x) or np.isinf(x))):
        return "N/A"
    return f"{100*x:.{casas}f}%"

def fmt_num(x, casas=3):
    if x is None or (isinstance(x, float) and (np.isnan(x) or np.isinf(x))):
        return "N/A"
    return f"{x:.{casas}f}"

def valor_seguro(df, filtro, coluna, default=np.nan):
    try:
        linha = df.loc[filtro, coluna]
        if len(linha) == 0:
            return default
        return linha.iloc[0]
    except Exception:
        return default

print("OK")

# ----------------------------------------------------------
# [3/6] METADADOS DINÂMICOS A PARTIR DAS ETAPAS 5 E 6
# ----------------------------------------------------------
print("\n[3/6] Preparando metadados dinâmicos...")

tabela_estatisticas_local = globals().get("tabela_estatisticas", pd.DataFrame()).copy()
tabela_testes_zero_local = globals().get("tabela_testes_zero", pd.DataFrame()).copy()
tabela_comparacoes_pareadas_local = globals().get("tabela_comparacoes_pareadas", pd.DataFrame()).copy()
df_janelas_resumo_local = globals().get("df_janelas_resumo", pd.DataFrame()).copy()

# Fallback: se não existirem em memória, tentamos trabalhar de forma mais genérica
tem_est = not tabela_estatisticas_local.empty
tem_zero = not tabela_testes_zero_local.empty
tem_pareado = not tabela_comparacoes_pareadas_local.empty
tem_df = not df_janelas_resumo_local.empty

# Métricas da janela principal
n_anos = np.nan
media_dez_jan = np.nan
mediana_dez_jan = np.nan
pct_positivo_dez_jan = np.nan
p_t_dez_jan = np.nan
p_perm_dez_jan = np.nan

if tem_est:
    n_anos = valor_seguro(
        tabela_estatisticas_local,
        tabela_estatisticas_local["janela"].astype(str) == "dez_jan",
        "n_anos"
    )
    media_dez_jan = valor_seguro(
        tabela_estatisticas_local,
        tabela_estatisticas_local["janela"].astype(str) == "dez_jan",
        "media_spread"
    )
    mediana_dez_jan = valor_seguro(
        tabela_estatisticas_local,
        tabela_estatisticas_local["janela"].astype(str) == "dez_jan",
        "mediana_spread"
    )
    pct_positivo_dez_jan = valor_seguro(
        tabela_estatisticas_local,
        tabela_estatisticas_local["janela"].astype(str) == "dez_jan",
        "pct_spread_positivo"
    )

if tem_zero:
    p_t_dez_jan = valor_seguro(
        tabela_testes_zero_local,
        (tabela_testes_zero_local["janela"].astype(str) == "dez_jan")
        & (tabela_testes_zero_local["teste"].astype(str) == "t-test unilateral"),
        "p_valor"
    )
    p_perm_dez_jan = valor_seguro(
        tabela_testes_zero_local,
        (tabela_testes_zero_local["janela"].astype(str) == "dez_jan")
        & (tabela_testes_zero_local["teste"].astype(str) == "permutação unilateral"),
        "p_valor"
    )

# Melhor comparação pareada
melhor_comp = None
if tem_pareado:
    tabela_comparacoes_pareadas_local = tabela_comparacoes_pareadas_local.sort_values(
        ["wilcoxon_p_valor", "permutacao_p_valor"],
        ascending=[True, True]
    ).reset_index(drop=True)
    if len(tabela_comparacoes_pareadas_local) > 0:
        melhor_comp = tabela_comparacoes_pareadas_local.iloc[0].to_dict()

if melhor_comp is not None:
    melhor_janela_controle = str(melhor_comp.get("janela_controle", "N/A"))
    melhor_media_diff = melhor_comp.get("media_diff_spread", np.nan)
    melhor_p_w = melhor_comp.get("wilcoxon_p_valor", np.nan)
    melhor_p_w_holm = melhor_comp.get("wilcoxon_p_holm", np.nan)
    melhor_p_perm = melhor_comp.get("permutacao_p_valor", np.nan)
    melhor_p_perm_holm = melhor_comp.get("permutacao_p_holm", np.nan)
else:
    melhor_janela_controle = "N/A"
    melhor_media_diff = np.nan
    melhor_p_w = np.nan
    melhor_p_w_holm = np.nan
    melhor_p_perm = np.nan
    melhor_p_perm_holm = np.nan

print(f"n_anos.........................: {n_anos}")
print(f"media_dez_jan.................: {fmt_num(media_dez_jan)}")
print(f"pct_positivo_dez_jan..........: {fmt_pct(pct_positivo_dez_jan)}")
print(f"p_t_dez_jan...................: {fmt_num(p_t_dez_jan, 4)}")
print(f"p_perm_dez_jan................: {fmt_num(p_perm_dez_jan, 4)}")
print(f"melhor_janela_controle........: {melhor_janela_controle}")
print(f"melhor_media_diff.............: {fmt_num(melhor_media_diff)}")
print(f"melhor_p_w....................: {fmt_num(melhor_p_w, 4)}")
print(f"melhor_p_w_holm...............: {fmt_num(melhor_p_w_holm, 4)}")
print(f"melhor_p_perm..................: {fmt_num(melhor_p_perm, 4)}")
print(f"melhor_p_perm_holm............: {fmt_num(melhor_p_perm_holm, 4)}")
print("OK")

# ----------------------------------------------------------
# [4/6] CONTEÚDO DAS SEÇÕES
# ----------------------------------------------------------
print("\n[4/6] Montando seções do HTML...")

# =============================
# RESUMO EXECUTIVO
# =============================
conteudo_resumo = f"""
<section class="executive-summary">
    <div class="exec-main">
        <h2>Resumo Executivo</h2>
        <p>
            Este projeto investiga a existência do <b>efeito janeiro</b> no mercado acionário brasileiro,
            comparando o desempenho relativo entre ações de <b>menor liquidez</b> e <b>maior liquidez</b>.
        </p>
        <p>
            A liquidez é utilizada como <b>proxy de tamanho</b>, e não como market cap observado diretamente.
            O foco central do estudo é verificar se a janela <b>dezembro→janeiro</b> apresenta um spread
            small-minus-large economicamente relevante e como esse comportamento se compara a janelas de controle
            em outros períodos do ano.
        </p>

        <div class="highlight-grid">
            <div class="highlight-card">
                <span class="label">Proxy utilizada</span>
                <div class="value">Liquidez média de novembro</div>
            </div>
            <div class="highlight-card">
                <span class="label">Recorte dos grupos</span>
                <div class="value">Bottom 30% vs Top 30%</div>
            </div>
            <div class="highlight-card">
                <span class="label">Janela principal</span>
                <div class="value">Dezembro → Janeiro</div>
            </div>
            <div class="highlight-card">
                <span class="label">Anos avaliados</span>
                <div class="value">{int(n_anos) if pd.notna(n_anos) else "N/A"}</div>
            </div>
        </div>
    </div>

    <div class="exec-side">
        <h2>Key Takeaways</h2>
        <div class="takeaways">
            <div class="takeaway-card">
                <h3>Maior spread médio na virada do ano</h3>
                <p>
                    A janela dezembro→janeiro apresentou spread médio de <b>{fmt_num(media_dez_jan)}</b>,
                    positivo em <b>{fmt_pct(pct_positivo_dez_jan)}</b> dos anos analisados.
                </p>
            </div>
            <div class="takeaway-card">
                <h3>Sinal estatístico favorável contra zero</h3>
                <p>
                    Na janela principal, os testes contra zero resultaram em
                    <b>p_t={fmt_num(p_t_dez_jan, 4)}</b> e
                    <b>p_perm={fmt_num(p_perm_dez_jan, 4)}</b>.
                </p>
            </div>
            <div class="takeaway-card">
                <h3>Comparação com controles mais fraca</h3>
                <p>
                    A comparação mais favorável foi contra <b>{html.escape(melhor_janela_controle)}</b>,
                    com diferença média de <b>{fmt_num(melhor_media_diff)}</b>, mas sem robustez após ajuste de Holm.
                </p>
            </div>
            <div class="takeaway-card">
                <h3>Evidência mais plausível que definitiva</h3>
                <p>
                    Os resultados sustentam um padrão economicamente interessante para a janela dezembro→janeiro,
                    mas não uma superioridade estatística forte e estável sobre todas as janelas de controle.
                </p>
            </div>
        </div>
    </div>
</section>
"""

# =============================
# INTRODUÇÃO
# =============================
conteudo_introducao = f"""
<div class="texto-livre">
    <p>
        O chamado <b>efeito janeiro</b> é uma das anomalias sazonais mais conhecidas da literatura financeira.
        Em linhas gerais, a hipótese sugere que ações menores ou menos negociadas tendem a apresentar
        desempenho relativamente superior na virada do ano.
    </p>

    <p>
        Como esta base não utiliza market cap histórico consolidado para todo o período, o estudo adota
        a <b>liquidez média de novembro</b> como <b>proxy de tamanho</b>. Assim, a leitura correta do projeto
        não é “small caps medidas diretamente por capitalização”, mas sim uma comparação entre
        <b>ações de menor liquidez</b> e <b>ações de maior liquidez</b>.
    </p>

    <p>
        O objetivo central não é apenas verificar se o spread small-minus-large é positivo em
        dezembro→janeiro, mas também se essa janela se destaca frente a janelas de controle
        como março→abril, abril→maio, julho→agosto e agosto→setembro.
    </p>
</div>
"""

# =============================
# METODOLOGIA
# =============================
conteudo_metodologia = f"""
<div class="texto-livre">
    <p>
        A base foi construída com dados históricos por ticker, contendo data, preço ajustado e volume financeiro.
        Para cada ano da amostra, foi calculada a <b>liquidez média de novembro</b>, utilizada como critério de classificação dos ativos.
    </p>

    <ul>
        <li><b>Small caps proxy por liquidez:</b> bottom 30% em liquidez</li>
        <li><b>Large caps proxy por liquidez:</b> top 30% em liquidez</li>
        <li><b>Faixa central:</b> excluída para aumentar o contraste entre os grupos</li>
    </ul>

    <p>
        A janela principal foi definida entre o primeiro pregão após meados de dezembro e o último pregão útil de janeiro.
        Além disso, foram construídas janelas de controle ao longo do ano para avaliar se o comportamento de dezembro→janeiro
        se destaca em relação a outros períodos.
    </p>

    <p>
        A análise estatística foi estruturada em <b>duas camadas</b>:
    </p>

    <ul>
        <li><b>Testes contra zero</b>, avaliando se o spread da janela é positivo;</li>
        <li><b>Testes pareados</b>, comparando <b>dez_jan</b> com cada janela de controle ano a ano.</li>
    </ul>

    <p>
        Foram utilizados:
    </p>

    <ul>
        <li><b>Teste t unilateral</b> e <b>teste de permutação</b> contra zero;</li>
        <li><b>Wilcoxon pareado unilateral</b> e <b>permutação pareada</b> para a hipótese central;</li>
        <li><b>Ajuste de Holm</b> nas comparações múltiplas contra as janelas de controle.</li>
    </ul>

    <p>
        Esse desenho permite separar duas perguntas diferentes:
        se a janela principal é positiva em si mesma e se ela realmente se destaca em relação às janelas de controle.
    </p>
</div>
"""

# =============================
# TABELAS
# =============================
conteudo_tabelas = ""

conteudo_tabelas += """
<div class="subsection-block">
    <h3>Base e Formação das Carteiras</h3>
</div>
"""
conteudo_tabelas += bloco_tabelas_html([
    ("tabela_amostra_anual", "Amostra Anual de Ativos Elegíveis"),
    ("tabela_retorno_carteiras", "Retornos das Carteiras na Janela Principal"),
    ("tabela_base_consolidada_janelas", "Base Consolidada das Janelas"),
    ("tabela_comparacao_janelas", "Comparação entre Janelas"),
])

conteudo_tabelas += """
<div class="subsection-block">
    <h3>Estatísticas e Hipótese Central</h3>
</div>
"""
conteudo_tabelas += bloco_tabelas_html([
    ("tabela_resumo_estatistico", "Resumo Estatístico dos Spreads"),
    ("tabela_testes_contra_zero", "Testes Contra Zero por Janela"),
    ("tabela_comparacoes_pareadas_janela", "Comparações Pareadas: Dezembro→Janeiro vs Controles"),
])
conteudo_tabelas += bloco_tabela_preferencial(
    ["tabela_resumo_hipotese_central_visual", "tabela_resumo_hipotese_central"],
    "Resumo da Hipótese Central"
)
conteudo_tabelas += bloco_tabelas_html([
    ("tabela_testes_estatisticos", "Tabela Consolidada dos Testes"),
    ("tabela_spread_medio_por_janela", "Spread Médio por Janela"),
    ("tabela_spread_por_ano_e_janela", "Spread por Ano e Janela"),
    ("tabela_diferencas_pareadas_por_ano", "Diferenças Pareadas por Ano"),
])

# =============================
# GRÁFICOS
# =============================
conteudo_graficos = ""

conteudo_graficos += bloco_imagens_com_titulo(
    "Comparação Direta entre os Grupos",
    [
        ("grafico_retorno_medio_janelas", "Retorno Médio por Janela (Small vs Large)"),
    ]
)

conteudo_graficos += bloco_imagens_com_titulo(
    "Spreads e Distribuições",
    [
        ("boxplot_spread_janelas", "Distribuição do Spread por Janela"),
        ("linha_spread_tempo", "Evolução Temporal do Spread"),
        ("hist_spread_janeiro", "Distribuição do Spread em Dezembro→Janeiro"),
        ("grafico_spread_medio_por_janela", "Spread Médio Small-Minus-Large por Janela"),
        ("grafico_boxplot_spread_por_janela", "Boxplot do Spread por Janela"),
        ("grafico_spread_por_ano_e_janela", "Spread por Ano e Janela"),
    ]
)

conteudo_graficos += bloco_imagens_com_titulo(
    "Hipótese Central: Dezembro→Janeiro vs Janelas de Controle",
    [
        ("grafico_comparacao_pareada_dez_jan_vs_controles", "Comparação Pareada: Dezembro→Janeiro vs Controles"),
        ("grafico_boxplot_diferencas_pareadas", "Distribuição das Diferenças Pareadas"),
    ]
)

# =============================
# LEITURA DOS RESULTADOS
# =============================
conteudo_resultados = f"""
<div class="texto-livre">
    <p>
        O principal achado do projeto é que a janela <b>dezembro→janeiro</b> apresentou spread médio de <b>{fmt_num(media_dez_jan)}</b>,
        positivo em <b>{fmt_pct(pct_positivo_dez_jan)}</b> dos anos analisados.
        Isso sugere que ações de menor liquidez tenderam, em média, a superar
        ações de maior liquidez nessa virada de ano.
    </p>

    <p>
        Nos testes <b>contra zero</b>, a janela principal ficou com
        <b>p_t={fmt_num(p_t_dez_jan, 4)}</b> e
        <b>p_perm={fmt_num(p_perm_dez_jan, 4)}</b>,
        o que reforça que o spread de dezembro→janeiro não foi apenas visualmente positivo,
        mas também estatisticamente relevante nessa primeira camada de análise.
    </p>

    <p>
        Na comparação direta entre dezembro→janeiro e as janelas de controle ano a ano,
        a relação mais favorável apareceu contra <b>{html.escape(melhor_janela_controle)}</b>,
        com diferença média de <b>{fmt_num(melhor_media_diff)}</b>.
        Ainda assim, os valores ajustados por Holm ficaram em
        <b>{fmt_num(melhor_p_w_holm, 4)}</b> no Wilcoxon e
        <b>{fmt_num(melhor_p_perm_holm, 4)}</b> na permutação pareada.
    </p>

    <p>
        Isso sugere que <b>dezembro→janeiro parece economicamente mais forte</b>,
        mas a superioridade estatística dessa janela sobre as demais
        <b>não ficou robusta depois do teste pareado com ajuste por múltiplas comparações</b>.
    </p>

    <p>
        A leitura temporal do spread também merece atenção. O ano de <b>2020</b> aparece como um dos pontos
        mais fortes da série, elevando a média da janela principal. Como esse resultado corresponde à passagem
        de <b>dezembro de 2019 para janeiro de 2020</b>, ele antecede o período mais agudo da pandemia e,
        portanto, não deve ser interpretado como efeito direto do choque mais severo de COVID-19.
    </p>

    <p>
        Isso não invalida o resultado, mas mostra que parte da magnitude observada pode depender de episódios
        específicos da amostra. Em outras palavras, o conjunto dos resultados aponta para um padrão plausível,
        mas não para uma anomalia sazonal fortemente estável e incontestável.
    </p>

    <p>
        Também é importante destacar que a narrativa correta não é “small caps medidas diretamente por market cap”.
        O estudo utiliza <b>liquidez como proxy</b>, então a interpretação mais adequada é a de um possível
        efeito janeiro entre ações de <b>menor liquidez</b> versus ações de <b>maior liquidez</b>.
    </p>
</div>
"""

# =============================
# CONCLUSÃO
# =============================
conteudo_conclusao = f"""
<div class="conclusao">
    <h2>Conclusão</h2>

    <p>
        Este projeto mostrou que ações de <b>menor liquidez</b> tenderam a apresentar desempenho superior
        às de <b>maior liquidez</b> na janela entre meados de dezembro e o final de janeiro no mercado brasileiro.
        Em média, a janela principal apresentou spread de <b>{fmt_num(media_dez_jan)}</b>,
        com predominância de anos positivos.
    </p>

    <p>
        Quando a janela principal é comparada diretamente com as janelas de controle,
        o resultado final fica mais equilibrado:
        há <b>evidência econômica plausível</b> em favor do efeito janeiro,
        e a janela principal passa nos testes contra zero,
        mas a superioridade frente às janelas de controle
        <b>não ficou robusta depois dos testes pareados com ajuste de Holm</b>.
    </p>

    <p>
        A leitura da série ao longo do tempo também sugere cautela.
        O ano de <b>2020</b> aparece como um ponto de destaque e contribui para elevar a magnitude média
        observada em dezembro→janeiro. Como esse movimento corresponde à janela entre
        <b>dezembro de 2019 e janeiro de 2020</b>, ele antecede o período mais crítico da pandemia e
        deve ser lido como parte da dinâmica específica da amostra, não como evidência de um único choque isolado.
    </p>

    <p>
        Portanto, a conclusão mais correta não é afirmar uma anomalia forte e definitiva,
        mas sim reconhecer um padrão economicamente interessante, com suporte estatístico moderado
        e sensível ao rigor do critério de comparação.
    </p>

    <p>
        Além disso, a interpretação deve respeitar a proxy utilizada:
        o projeto fala de <b>menor liquidez como proxy de tamanho</b>,
        e não de small caps medidas diretamente por market cap.
        Isso não invalida o estudo, mas delimita com mais precisão o escopo da evidência encontrada.
    </p>

    <h3>Principais aprendizados</h3>
    <ul>
        <li>A janela <b>dezembro→janeiro</b> apresentou o maior spread médio do estudo.</li>
        <li>O spread da janela principal foi positivo em <b>{fmt_pct(pct_positivo_dez_jan)}</b> dos anos analisados.</li>
        <li>Nos testes contra zero, dezembro→janeiro apresentou evidência estatística favorável.</li>
        <li>A comparação pareada contra janelas de controle enfraqueceu a força da conclusão.</li>
        <li>A melhor comparação foi contra <b>{html.escape(melhor_janela_controle)}</b>, mas sem robustez após Holm.</li>
        <li>O ano de <b>2020</b> teve peso relevante na magnitude observada da janela principal.</li>
        <li>O estudo sustenta melhor uma <b>evidência econômica plausível</b> do que uma superioridade estatística definitiva.</li>
        <li>A leitura correta é feita em termos de <b>proxy por liquidez</b>, e não market cap observado diretamente.</li>
    </ul>
</div>
"""

print("Seções montadas.")
print("OK")

# ----------------------------------------------------------
# [5/6] MONTAGEM DO HTML
# ----------------------------------------------------------
print("\n[5/6] Construindo HTML final...")

html_final = f"""
<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Efeito Janeiro no Brasil</title>

    <link rel="stylesheet" href="https://cdn.datatables.net/1.13.8/css/jquery.dataTables.min.css">

    <style>
        body {{
            font-family: Arial, Helvetica, sans-serif;
            background: #f7f8fa;
            color: #1f2937;
            margin: 0;
            padding: 0;
        }}

        .container {{
            width: 92%;
            max-width: 1450px;
            margin: 0 auto;
            padding: 24px 0 48px 0;
        }}

        .hero {{
            background: linear-gradient(135deg, #111827, #1f2937);
            color: white;
            padding: 32px;
            border-radius: 16px;
            margin-bottom: 28px;
            box-shadow: 0 10px 25px rgba(0,0,0,0.15);
        }}

        .hero h1 {{
            margin: 0 0 12px 0;
            font-size: 2rem;
        }}

        .hero p {{
            margin: 6px 0;
            line-height: 1.6;
        }}

        .nav {{
            display: flex;
            flex-wrap: wrap;
            gap: 10px;
            margin-top: 20px;
        }}

        .nav a {{
            text-decoration: none;
            color: white;
            background: rgba(255,255,255,0.12);
            padding: 10px 14px;
            border-radius: 10px;
            font-size: 0.95rem;
        }}

        .section {{
            background: white;
            border-radius: 16px;
            padding: 24px;
            margin-bottom: 24px;
            box-shadow: 0 6px 18px rgba(0,0,0,0.08);
        }}

        .section h2 {{
            margin-top: 0;
            font-size: 1.5rem;
            color: #111827;
        }}

        .section-desc {{
            color: #4b5563;
            margin-bottom: 20px;
            line-height: 1.6;
        }}

        .texto-livre p,
        .conclusao p {{
            line-height: 1.8;
            color: #374151;
        }}

        .texto-livre ul,
        .conclusao ul {{
            line-height: 1.8;
            color: #374151;
            padding-left: 22px;
        }}

        .subsection-block {{
            margin-top: 10px;
            margin-bottom: 18px;
        }}

        .subsection-block h3 {{
            margin-bottom: 14px;
            color: #111827;
        }}

        .grid-2 {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(420px, 1fr));
            gap: 20px;
            margin-bottom: 20px;
        }}

        .chart-card {{
            background: #fafafa;
            border: 1px solid #e5e7eb;
            border-radius: 14px;
            padding: 16px;
        }}

        .chart-card h3 {{
            margin-top: 0;
            font-size: 1rem;
        }}

        .chart-card img {{
            width: 100%;
            border-radius: 10px;
            display: block;
            cursor: zoom-in;
            transition: transform 0.2s ease;
        }}

        .chart-card img:hover {{
            transform: scale(1.01);
        }}

        .table-block {{
            margin-top: 18px;
            margin-bottom: 18px;
        }}

        .table-block h3 {{
            margin-bottom: 10px;
        }}

        .table-wrapper {{
            overflow-x: auto;
            overflow-y: auto;
            max-height: 700px;
            border: 1px solid #e5e7eb;
            border-radius: 12px;
            background: #fff;
            padding: 8px;
        }}

        .footer {{
            text-align: center;
            color: #6b7280;
            font-size: 0.9rem;
            margin-top: 32px;
        }}

        .modal {{
            display: none;
            position: fixed;
            z-index: 9999;
            inset: 0;
            background: rgba(0, 0, 0, 0.88);
            justify-content: center;
            align-items: center;
            padding: 24px;
        }}

        .modal.show {{
            display: flex;
        }}

        .modal img {{
            max-width: 95%;
            max-height: 92vh;
            border-radius: 12px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.35);
        }}

        .modal-close {{
            position: absolute;
            top: 18px;
            right: 24px;
            color: white;
            font-size: 2rem;
            font-weight: bold;
            cursor: pointer;
            line-height: 1;
        }}

        .executive-summary {{
            display: grid;
            grid-template-columns: 1.4fr 1fr;
            gap: 20px;
            margin: 0 0 24px 0;
        }}

        .exec-main,
        .exec-side {{
            background: white;
            border-radius: 16px;
            padding: 24px;
            box-shadow: 0 6px 18px rgba(0,0,0,0.08);
        }}

        .exec-main h2,
        .exec-side h2 {{
            margin-top: 0;
            color: #111827;
        }}

        .exec-main p,
        .exec-side p {{
            color: #4b5563;
            line-height: 1.7;
        }}

        .highlight-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
            gap: 16px;
            margin-top: 18px;
        }}

        .highlight-card {{
            background: #f9fafb;
            border: 1px solid #e5e7eb;
            border-radius: 14px;
            padding: 16px;
        }}

        .highlight-card .label {{
            display: block;
            font-size: 0.85rem;
            color: #6b7280;
            margin-bottom: 8px;
        }}

        .highlight-card .value {{
            font-size: 1.05rem;
            font-weight: bold;
            color: #111827;
            line-height: 1.4;
        }}

        .takeaways {{
            display: grid;
            grid-template-columns: 1fr;
            gap: 16px;
            margin-top: 18px;
        }}

        .takeaway-card {{
            background: #111827;
            color: white;
            border-radius: 14px;
            padding: 18px;
            box-shadow: 0 6px 18px rgba(0,0,0,0.10);
        }}

        .takeaway-card h3 {{
            margin-top: 0;
            margin-bottom: 10px;
            font-size: 1rem;
        }}

        .takeaway-card p {{
            margin: 0;
            line-height: 1.6;
            color: rgba(255,255,255,0.88);
        }}

        @media (max-width: 980px) {{
            .executive-summary {{
                grid-template-columns: 1fr;
            }}
        }}
    </style>
</head>
<body>
    <div class="container">

        <div class="hero">
            <h1>Efeito Janeiro no Brasil</h1>
            <p>
                Uma análise empírica do desempenho relativo entre ações de menor liquidez e maior liquidez
                no mercado acionário brasileiro.
            </p>
            <p>
                O estudo utiliza liquidez como proxy de tamanho e combina evidência econômica,
                testes estatísticos e comparações com janelas de controle ao longo do ano.
            </p>

            <div class="nav">
                <a href="#introducao">Introdução</a>
                <a href="#metodologia">Metodologia</a>
                <a href="#tabelas">Tabelas</a>
                <a href="#graficos">Gráficos</a>
                <a href="#resultados">Resultados</a>
                <a href="#conclusao">Conclusão</a>
            </div>
        </div>

        {conteudo_resumo}

        <div id="introducao">
            {secao_html("Introdução", "Contexto do efeito janeiro, proxy utilizada e objetivo da análise.", conteudo_introducao)}
        </div>

        <div id="metodologia">
            {secao_html("Metodologia", "Construção das carteiras, definição das janelas e testes utilizados.", conteudo_metodologia)}
        </div>

        <div id="tabelas">
            {secao_html("Tabelas", "Tabelas principais da amostra, das janelas, dos testes e das comparações pareadas.", conteudo_tabelas)}
        </div>

        <div id="graficos">
            {secao_html("Gráficos", "Comparação entre os grupos, comportamento dos spreads e evidência visual da hipótese central.", conteudo_graficos)}
        </div>

        <div id="resultados">
            {secao_html("Leitura dos Resultados", "Interpretação econômica e estatística dos achados do estudo.", conteudo_resultados)}
        </div>

        <div id="conclusao">
            {conteudo_conclusao}
        </div>

        <div class="footer">
            <p>HTML gerado automaticamente a partir das saídas do projeto.</p>
        </div>
    </div>

    <div id="imgModal" class="modal">
        <span class="modal-close" id="modalClose">&times;</span>
        <img id="modalImg" src="" alt="Imagem ampliada">
    </div>

    <script>
        const modal = document.getElementById("imgModal");
        const modalImg = document.getElementById("modalImg");
        const modalClose = document.getElementById("modalClose");

        document.querySelectorAll(".zoomable-img").forEach(img => {{
            img.addEventListener("click", () => {{
                modalImg.src = img.src;
                modalImg.alt = img.alt;
                modal.classList.add("show");
            }});
        }});

        modalClose.addEventListener("click", () => {{
            modal.classList.remove("show");
            modalImg.src = "";
        }});

        modal.addEventListener("click", (e) => {{
            if (e.target === modal) {{
                modal.classList.remove("show");
                modalImg.src = "";
            }}
        }});

        document.addEventListener("keydown", (e) => {{
            if (e.key === "Escape") {{
                modal.classList.remove("show");
                modalImg.src = "";
            }}
        }});
    </script>

    <script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
    <script src="https://cdn.datatables.net/1.13.8/js/jquery.dataTables.min.js"></script>

    <script>
        $(document).ready(function () {{
            $("table").DataTable({{
                paging: false,
                info: true,
                searching: true,
                ordering: true,
                lengthChange: false,
                order: [],
                language: {{
                    decimal: ",",
                    thousands: ".",
                    zeroRecords: "Nenhum registro encontrado",
                    info: "Mostrando _TOTAL_ registros",
                    infoEmpty: "Mostrando 0 registros",
                    infoFiltered: "(filtrado de _MAX_ registros no total)",
                    search: "Filtrar:",
                    paginate: {{
                        first: "Primeiro",
                        last: "Último",
                        next: "Próximo",
                        previous: "Anterior"
                    }}
                }}
            }});
        }});
    </script>
</body>
</html>
"""

print("HTML construído.")
print("OK")

# ----------------------------------------------------------
# [6/6] SALVAR HTML
# ----------------------------------------------------------
print("\n[6/6] Salvando HTML...")

HTML_PATH = BASE_DIR / "index.html"

with open(HTML_PATH, "w", encoding="utf-8") as f:
    f.write(html_final)

print(f"HTML salvo em: {HTML_PATH}")

print("\n" + "=" * 70)
print("ETAPA 7 CONCLUÍDA")
print("=" * 70)

ETAPA 7 - HTML FINAL

[1/6] Validando ambiente...
BASE_DIR......: C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\efeito_janeiro_small_caps_brasil
GRAFICOS_DIR..: C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\efeito_janeiro_small_caps_brasil\resultados\graficos
TABELAS_DIR...: C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\efeito_janeiro_small_caps_brasil\resultados\tabelas
OK

[2/6] Definindo funções auxiliares...
OK

[3/6] Preparando metadados dinâmicos...
n_anos.........................: 15.0
media_dez_jan.................: 0.035
pct_positivo_dez_jan..........: 66.7%
p_t_dez_jan...................: 0.0356
p_perm_dez_jan................: 0.0363
melhor_janela_controle........: mar_abr
melhor_media_diff.............: 0.037
melhor_p_w....................: 0.0473
melhor_p_w_holm...............: 0.1892
melhor_p_perm..................: 0.0792
melhor_p_perm_holm............: 0.3168
OK

[4/6] Montando seções do HTML...
Seções montadas.
OK

[5/6] Construindo HTML final...
HTML constr

# Conclusão

Este projeto investigou se ações de **menor liquidez** tendem a apresentar desempenho superior às de **maior liquidez** na janela entre meados de dezembro e o fim de janeiro no mercado brasileiro.

Os resultados mostraram que a janela **dezembro→janeiro** apresentou o **maior spread médio** entre todas as janelas analisadas, com valor médio de aproximadamente **3,5%**, além de spread positivo em **cerca de dois terços dos anos** da amostra. Esse resultado sustenta a leitura de que existe uma **evidência econômica plausível** de efeito janeiro quando os ativos são segmentados por liquidez.

Nos testes **contra zero**, a janela principal apresentou sinal estatístico favorável, com resultados consistentes entre o **teste t unilateral** e o **teste de permutação**. Isso reforça que o spread observado em dezembro→janeiro não foi apenas visualmente positivo, mas também estatisticamente relevante nessa primeira camada de análise.

No entanto, a hipótese mais forte do projeto não é apenas que dezembro→janeiro seja positiva, mas sim que ela seja **superior às janelas de controle**. Quando essa comparação foi feita de forma **pareada por ano**, a evidência ficou mais fraca. A relação mais favorável apareceu na comparação com **março→abril**, mas a robustez desaparece quando aplicamos o **ajuste de Holm** para múltiplas comparações. Em outras palavras, o estudo sugere que dezembro→janeiro **parece economicamente mais forte**, mas sua superioridade estatística sobre as demais janelas **não ficou robusta**.

A análise temporal também recomenda cautela. O ano de **2020** aparece como um dos pontos mais fortes da série e ajuda a elevar a média observada da janela principal. Como esse movimento corresponde à passagem de **dezembro de 2019 para janeiro de 2020**, ele antecede o período mais agudo da pandemia e não deve ser lido como reflexo direto do crash de COVID-19. Ainda assim, sua presença mostra que parte da magnitude observada pode depender de episódios específicos da amostra.

Outro ponto importante é a interpretação correta da proxy utilizada. O projeto não mede small caps diretamente por **market cap**, mas sim por **liquidez média de novembro**. Portanto, a leitura mais precisa não é a de uma prova definitiva sobre small caps no sentido estrito, e sim a de um possível efeito janeiro entre ações de **menor liquidez** versus ações de **maior liquidez**.

📊 Interpretação dos resultados

Os resultados permitem algumas conclusões importantes:

- A janela **dezembro→janeiro** apresentou o maior spread médio do estudo
- O spread da janela principal foi positivo na maior parte dos anos analisados
- Os testes **contra zero** mostraram evidência estatística favorável para a janela principal
- A comparação pareada com janelas de controle enfraqueceu a força da hipótese central
- A melhor comparação ocorreu contra **março→abril**, mas sem robustez após ajuste por múltiplas comparações
- O ano de **2020** teve peso relevante na magnitude observada do efeito
- A evidência é mais forte em termos **econômicos** do que em termos de **robustez estatística definitiva**

🧠 Principais aprendizados

- Ações de **menor liquidez** podem, sim, superar ações de maior liquidez na virada do ano
- A janela **dezembro→janeiro** foi a mais forte entre as janelas analisadas
- O efeito possui suporte estatístico quando testado **contra zero**
- A superioridade frente às janelas de controle **não ficou robusta**
- Parte da magnitude observada é sensível a anos específicos, especialmente **2020**
- A **liquidez** se mostrou uma proxy prática para segmentação, mas não equivale a market cap observado diretamente

🏁 Conclusão final

O estudo aponta para um padrão **economicamente interessante** e compatível com a hipótese do efeito janeiro no Brasil, mas não para uma anomalia sazonal **forte, estável e estatisticamente incontestável**.

Em termos práticos, a leitura mais correta é:

> ações de menor liquidez **podem** apresentar desempenho superior no início do ano,  
> mas esse comportamento deve ser tratado como uma **evidência plausível com robustez estatística moderada**,  
> e não como uma regularidade definitiva do mercado brasileiro.